<!-- Gebruiker
   ↓
Natuurlijke taal
   ↓
AI-agent
   ↓
Playwright Browser Toolkit
   ↓
Playwright
   ↓
Webbrowser -->

<!-- Stap 1: open je Jupyter-notebook in je ai4beg-omgeving en voer alleen dit uit: Stap 1 — Controleren welke Python-omgeving Jupyter gebruikt

Waarom: We willen zeker weten dat je notebook draait in dezelfde ai4beg-omgeving waarin we de benodigde packages installeren. Anders kan het gebeuren dat een package wel is geïnstalleerd, maar Jupyter hem niet kan vinden.

We gebruikten:-->

In [1]:
import sys
print(sys.executable)

c:\Users\Mill\agentsforQA\.venv312\Scripts\python.exe


<!-- Stap 2 van les 46: controleer of nest_asyncio aanwezig is. Voer alleen dit uit: -->

In [2]:
import nest_asyncio

ModuleNotFoundError: No module named 'nest_asyncio'

<!-- Stap 3 — nest_asyncio activeren

Waarom: Jupyter draait al een asyncio event loop. Playwright werkt met asynchrone functies. nest_asyncio maakt het mogelijk om die async Playwright-code vanuit je notebook te gebruiken zonder problemen met de bestaande event loop.

Voer uit: -->

In [ ]:
nest_asyncio.apply()

<!-- Stap 4a — langchain-community installeren

Waarom: De PlaywrightBrowserToolkit die we uit de cursus willen gebruiken zit in de LangChain Community-integraties. Daarom moet deze package eerst in jouw huidige ai4beg-omgeving aanwezig zijn.

Voer in een nieuwe Jupyter-cel uit: -->

In [ ]:
%pip install langchain-community

<!-- Stap 4b — Kernel herstarten

Waarom: Jupyter had de oude Python-omgeving al geladen voordat we langchain-community installeerden. Door de kernel te herstarten zorgt Jupyter ervoor dat de nieuwe package daadwerkelijk beschikbaar komt.

Herstart nu de kernel via:

Kernel → Restart Kernel

Daarna zijn de eerder uitgevoerde imports en variabelen verdwenen. Dat is normaal.

Na de restart voeren we eerst opnieuw stap 1 uit om te controleren dat we nog steeds in ai4beg zitten: -->

In [ ]:
import sys
print(sys.executable)

<!-- Stap 5 — nest_asyncio opnieuw laden

Waarom: Na de kernelrestart is de Python-sessie volledig opnieuw gestart. Daardoor zijn ook onze eerdere imports verdwenen. We laden nest_asyncio opnieuw voordat we verdergaan met de asynchrone Playwright-code.

Voer uit: -->

In [ ]:
import nest_asyncio
nest_asyncio.apply()

<!-- Stap 6a — Controleren waar de Playwright-toolkit zit

Waarom: We willen eerst de exacte API van jouw geïnstalleerde langchain-community 0.4.2 vaststellen. Zo voorkomen we dat we blind verschillende imports proberen of je omgeving onnodig aanpassen.

Voer alleen dit uit: -->

In [ ]:
import langchain_community.agent_toolkits.playwright as pw
print(pw)

<!-- Stap 6c — Eerst controleren wat jouw Playwright-module daadwerkelijk bevat

Waarom: Voordat we packages gaan downgraden of een andere installatie kiezen, kijken we exact welke namen jouw huidige module aanbiedt. Daarmee kunnen we bepalen welke route het dichtst bij de cursus blijft.

Voer alleen dit uit: -->

In [ ]:
import langchain_community.agent_toolkits.playwright as pw

print(dir(pw))

<!-- Stap 6d — Juiste import

Waarom: We hebben nu rechtstreeks vastgesteld welke class jouw geïnstalleerde versie daadwerkelijk aanbiedt. We gebruiken daarom exact die naam.

Voer uit: -->

In [ ]:
from langchain_community.agent_toolkits.playwright import PlayWrightBrowserToolkit

<!-- Stap 7 — Async Playwright importeren

Waarom: De toolkit moet straks gekoppeld worden aan een Playwright-browser. De cursus gebruikt hiervoor de asynchrone Playwright-versie, zodat browseracties zoals navigeren en elementen ophalen met await kunnen worden uitgevoerd.

Voer uit: -->

In [ ]:
from playwright.async_api import async_playwright

<!-- Stap 8 — Playwright starten

Waarom: We hebben nu de Playwright-API en de toolkit geïmporteerd. Met async_playwright() starten we de Playwright-engine, die vervolgens een Chromium-browser kan openen. We maken hem eerst alleen aan; de browser zelf starten we in de volgende stap.

Voer uit: -->

Stap 8a — Vaststellen welk asyncio event loop Jupyter gebruikt

Waarom: Playwright moet een subprocess starten (create_subprocess_exec). Op Windows ondersteunt jouw huidige Jupyter-event-loop dat niet. We controleren eerst welke event loop actief is voordat we hem aanpassen.

In [ ]:
import asyncio

loop = asyncio.get_running_loop()

print(type(loop))
print(type(loop).__module__)

<!-- Stap 8b — Waarom Playwright nu faalt

Waarom: Je Jupyter-kernel gebruikt op Windows een WindowsSelectorEventLoop. Die event loop ondersteunt het starten van subprocesses niet op de manier waarop Playwright dat nodig heeft. Daarom kregen we:

NotImplementedError

We gaan dit niet oplossen door allerlei packages opnieuw te installeren. Playwright zelf werkt; het probleem zit in de manier waarop Jupyter de async event loop uitvoert.

Omdat de cursus juist async Playwright in Jupyter gebruikt, moeten we eerst de event-loop situatie correct oplossen.

Voer nu alleen dit uit: -->

In [ ]:
import asyncio

print(asyncio.get_event_loop_policy())

<!-- Stap 8c — Event-loop policy aanpassen

Waarom: We hebben vastgesteld dat Windows momenteel de WindowsSelectorEventLoopPolicy gebruikt. Playwright heeft voor het starten van zijn subprocess een event loop nodig die subprocesses op Windows ondersteunt. We schakelen daarom over naar de WindowsProactorEventLoopPolicy.

Voer dit uit: -->

In [ ]:
asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

In [ ]:
print(asyncio.get_event_loop_policy())

<!-- Stap 8d — Controleren welke event loop nu actief is

Waarom: De policy is nu correct aangepast. We moeten controleren of de Jupyter-kernel daadwerkelijk een ProactorEventLoop gebruikt voordat we Playwright opnieuw starten. -->

In [ ]:
loop = asyncio.get_event_loop()

print(type(loop))
print(type(loop).__module__)

Stap 8e — Playwright opnieuw starten

Waarom: Nu gebruikt je Jupyter-kernel daadwerkelijk de ProactorEventLoop, die op Windows subprocesses ondersteunt. Daarmee zou de fout uit stap 8 moeten zijn opgelost.

Gebruik opnieuw:

In [ ]:
loop = asyncio.get_running_loop()

print(type(loop))
print(type(loop).__module__)

In [ ]:
import nest_asyncio

print(nest_asyncio)
print(asyncio.get_running_loop())

In [ ]:
import asyncio
import threading
from playwright.async_api import async_playwright

result = {}

def run_playwright():
    async def main():
        async with async_playwright() as p:
            result["status"] = "Async Playwright OK"

    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)

    try:
        loop.run_until_complete(main())
    finally:
        loop.close()

thread = threading.Thread(target=run_playwright)
thread.start()
thread.join()

print(result)

Stap 12 — De Playwright-browser daadwerkelijk starten

Waarom: We weten nu dat we Playwright vanuit een aparte Proactor-event-loop kunnen starten. Nu testen we de volgende stap van de cursus: daadwerkelijk een Chromium-browser openen.

Voer deze cel uit:

In [ ]:
result = {}

def run_browser():
    async def main():
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            result["status"] = "Chromium OK"
            await browser.close()

    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)

    try:
        loop.run_until_complete(main())
    finally:
        loop.close()

thread = threading.Thread(target=run_browser)
thread.start()
thread.join()

print(result)

Stap 13 — Een browserobject beschikbaar maken voor de Toolkit

Waarom: De cursus gaat nu de Playwright-browser koppelen aan PlayWrightBrowserToolkit. De toolkit heeft een bestaand async browserobject nodig. We moeten daarom de browser niet meteen sluiten zoals bij de vorige test, maar hem beschikbaar houden.

Voer deze cel uit:

In [ ]:
browser_result = {}

def start_browser():
    async def main():
        p = await async_playwright().start()
        browser = await p.chromium.launch(headless=True)

        browser_result["playwright"] = p
        browser_result["browser"] = browser
        browser_result["loop"] = asyncio.get_running_loop()

    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)

    thread = threading.current_thread()
    browser_result["thread"] = thread

    loop.run_until_complete(main())

Stap 13a — Eerst controleren

Waarom: We willen voorkomen dat we meerdere Playwright-processen tegelijk starten. We controleren daarom alleen of browser_result momenteel leeg is.

Voer uit:

In [ ]:
print(browser_result)

Stap 14 — De cursusomgeving aanpassen

Waarom: De cursus verwacht dat await async_playwright().start() rechtstreeks vanuit Jupyter werkt. Op jouw Windows-installatie blijft Jupyter echter een SelectorEventLoop gebruiken. We moeten daarom de notebook op een manier uitvoeren waarbij Playwright zijn async subprocess kan starten.

De meest betrouwbare route is om de cursusnotebook via JupyterLab met een andere kernelconfiguratie te draaien. Maar voordat we iets veranderen, wil ik eerst controleren welke versie van Jupyter/IPython je gebruikt.

Voer uit:

In [ ]:
import IPython
import jupyterlab

print("IPython:", IPython.__version__)
print("JupyterLab:", jupyterlab.__version__)

Stap 15 — Controleren welke Toolkit-class we daadwerkelijk hebben

Waarom: We weten inmiddels dat jouw versie de class PlayWrightBrowserToolkit heet. Nu willen we zien welke constructor en methodes jouw geïnstalleerde versie aanbiedt. Daarmee kunnen we de cursuscode aanpassen zonder te gokken.

Voer alleen dit uit:

In [ ]:
import inspect

print(inspect.signature(PlayWrightBrowserToolkit))
print(inspect.signature(PlayWrightBrowserToolkit.from_browser))

Stap 16 — Een permanente Playwright-thread maken

Waarom: De browser moet verbonden blijven met zijn eigen ProactorEventLoop. Als de thread of loop stopt, wordt de browser onbruikbaar. Daarom maken we nu één permanente thread waarin Playwright en Chromium blijven draaien.

Voer deze cel uit:

In [ ]:
import asyncio
import threading

playwright_state = {}

def playwright_worker():
    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)

    async def start():
        p = await async_playwright().start()
        browser = await p.chromium.launch(headless=True)

        playwright_state["playwright"] = p
        playwright_state["browser"] = browser
        playwright_state["loop"] = loop

    loop.run_until_complete(start())
    loop.run_forever()

playwright_thread = threading.Thread(
    target=playwright_worker,
    daemon=True
)

playwright_thread.start()

Stap 1 — controleren of de browser actief is

Waarom: we controleren eerst of Playwright daadwerkelijk in de aparte Proactor-thread draait. Als dit goed is, kunnen we daarna de PlayWrightBrowserToolkit eraan koppelen.

Voer alleen dit uit:

In [ ]:
import time

time.sleep(1)

print(playwright_state)

Stap 2 — Playwright Toolkit koppelen

Waarom: in les 46 koppelt de docent de actieve browser aan PlayWrightBrowserToolkit. Daarmee krijgen we straks de Playwright-tools zoals navigate_browser, click_element, extract_text, enzovoort.

Voer alleen dit uit:

In [ ]:
from langchain_community.agent_toolkits.playwright import PlayWrightBrowserToolkit

toolkit = PlayWrightBrowserToolkit.from_browser(
    async_browser=playwright_state["browser"]
)

print(toolkit)

Stap 3 — alle beschikbare tools ophalen

Waarom: in les 47 laat de docent zien welke tools de PlayWrightBrowserToolkit beschikbaar stelt. Dit is de basis voor de volgende lessen.

Voer uit:

In [ ]:
tools = toolkit.get_tools()

print(f"Aantal tools: {len(tools)}")

for tool in tools:
    print(tool.name)

Precies. Dit komt overeen met les 47: de Toolkit bevat 7 tools.

We kunnen nu naar les 48: Working with Playwright ToolKit tools to navigate and get elements from UI.

Stap 4 — navigate_browser ophalen

Waarom: in les 48 gebruikt de docent eerst de navigate_browser-tool om met Playwright naar een webpagina te gaan.

Voer uit:

In [ ]:
navigate_tool = next(
    tool for tool in tools
    if tool.name == "navigate_browser"
)

print(navigate_tool)

Stap 5 — naar de EAApp-pagina navigeren

Waarom: dit is precies wat de docent in les 48 doet: de navigate_browser-tool aanroepen met de URL van de webapplicatie.

Gebruik:


Stap 5 opnieuw — eerst de navigatie uitvoeren op de juiste Playwright-loop

Waarom: onze browser draait in playwright_state["loop"]. We moeten daarom de navigate_browser-actie op precies die loop uitvoeren.

Voer alleen dit uit:

Stap 6 — controleer of de navigatie überhaupt op de Playwright-loop terechtkomt

Waarom: pending betekent dat navigate_tool.arun() niet terugkomt. Eerst controleren we of onze achtergrondloop nog daadwerkelijk draait.

Voer alleen dit uit:

In [ ]:
print(playwright_state["loop"].is_running())

Stap 7 — testen of de browser zelf nog reageert

Waarom: hiermee scheiden we Playwright van de LangChain-tool. Als dit werkt, weten we dat de browser gezond is en dat de blokkade in de Toolkit-aanroep zit.

Voer dit uit:

In [ ]:
async def get_page_title():
    browser = playwright_state["browser"]

    if browser.contexts:
        context = browser.contexts[0]
        page = context.pages[0] if context.pages else await context.new_page()
    else:
        page = await browser.new_page()

    return await page.title()


future = asyncio.run_coroutine_threadsafe(
    get_page_title(),
    playwright_state["loop"]
)

print(future.result(timeout=10))

Stap 8 — get_elements ophalen

Waarom: de docent gebruikt daarna get_elements om elementen uit de Employee-pagina te halen.

Voer alleen dit uit:

In [ ]:
get_elements_tool = next(
    tool for tool in tools
    if tool.name == "get_elements"
)

print(get_elements_tool)

Stap 9 — de tabelcellen ophalen

Waarom: de docent gebruikt een CSS-selector om de inhoud van de Employee-tabel te lezen. We gebruiken hier dezelfde aanpak als in de les.

Voer uit:

In [ ]:
future = asyncio.run_coroutine_threadsafe(
    get_elements_tool.arun(
        "td",
        ["inner_text"]
    ),
    playwright_state["loop"]
)

result = future.result(timeout=30)

print(result)

Goed. Dit werkt nu precies zoals bedoeld: get_elements haalt de gegevens uit de Employee-tabel op.

Je krijgt per medewerker de velden terug in deze volgorde: naam, leeftijd, salaris, duur, niveau, e-mail en de Details-knop.

We zijn hiermee inhoudelijk door het belangrijkste deel van les 48 heen. De volgende stap in de cursus is les 49: Building AI Agent for Web Scraping with Playwright Tools.

We hoeven nu niet verder te experimenteren met get_elements.

###### 49

Dan gaan we naar les 49: Building AI Agent for Web Scraping with Playwright Tools.

Stap 1 — de lokale LLM laden

Waarom: in deze les geeft de docent de Playwright-tools aan een AI-agent. De LLM bepaalt vervolgens zelf welke browser-tool nodig is. We gebruiken jouw werkende gemma4:latest.

Voer uit:

In [ ]:
try:
    from langchain_ollama import ChatOllama
except ModuleNotFoundError:
    %pip install langchain-ollama
    from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

print(llm)

Goed. De LLM is correct geladen.

Stap 2 — de AI-agent maken met de Playwright-tools

Waarom: dit is de kern van les 49. We geven de agent de 7 Playwright-tools. De agent kan dan zelf bepalen wanneer hij bijvoorbeeld navigate_browser, get_elements of extract_text moet gebruiken.

Omdat jouw LangChain-versie 1.4.2 gebruikt, komt create_agent uit langchain.agents.

Voer uit:

In [ ]:
try:
    from langchain.agents import create_agent
except ModuleNotFoundError:
    %pip install "langchain>=1.0,<2.0"
    from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools
)

print(agent)

Goed. De agent is aangemaakt. CompiledStateGraph is de verwachte output van create_agent() in LangChain 1.x.

Stap 3 — de agent een eerste web-scraping opdracht geven

Waarom: nu testen we het belangrijkste onderdeel van les 49: de agent krijgt een natuurlijke-taalopdracht en bepaalt zelf welke Playwright-tool(s) hij moet gebruiken.

Gebruik eerst een eenvoudige opdracht:

Stap 5 — eerst controleren welke input de agent verwacht

Waarom: LangChain 1.4.2 gebruikt een CompiledStateGraph. We willen eerst zien welke state/schema de agent heeft voordat we de agent opnieuw aanroepen. Dit voorkomt dat we blijven gokken met ainvoke().

Voer alleen dit uit:

In [ ]:
print(agent.input_schema.model_json_schema())

In [ ]:
response = llm.invoke(
    "Reply with exactly: Hello"
)

print(response.content)

Stap 1 — testen of Gemma überhaupt een tool-call kan genereren.

Waarom: create_agent moet eerst aan het model vragen welke tool het moet gebruiken. Als Gemma daar vastloopt, weten we dat vóórdat Playwright wordt uitgevoerd.

Voer dit uit:

In [ ]:
llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(
    "Go to http://eaapp.somee.com/Employee"
)

print(response)

Stap 2 — de tool-call handmatig uitvoeren.

Waarom: hiermee testen we de exacte tool die de agent probeert te gebruiken, maar zonder create_agent. We weten al dat get_elements via onze Proactor-loop werkt, dus nu controleren we navigate_browser afzonderlijk.

Voer dit uit:

In [ ]:
future = asyncio.run_coroutine_threadsafe(
    navigate_tool.arun(
        "http://eaapp.somee.com/Employee"
    ),
    playwright_state["loop"]
)

result = future.result(timeout=30)

print(result)

Stap 3 — we testen nu de volledige agent opnieuw, maar met dezelfde URL en met een korte opdracht.

Waarom: nu de afzonderlijke onderdelen werken, moeten we controleren of de agent inmiddels de volledige cyclus kan uitvoeren: LLM → navigate_browser → resultaat → LLM.

Gebruik:

Stap 4 — een eenvoudige testtool gebruiken

Waarom: hiermee scheiden we create_agent van de Playwright/Jupyter-eventloop. Als deze simpele tool werkt, weten we dat create_agent zelf goed is en dat de combinatie met de async Playwright-tool het probleem veroorzaakt.

Voer eerst dit uit:

In [ ]:
from langchain.tools import tool

@tool
def test_tool(message: str) -> str:
    """Returns the supplied message."""
    return f"Tool received: {message}"

In [ ]:
test_agent = create_agent(
    model=llm,
    tools=[test_tool]
)

In [ ]:
response = test_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Use the test_tool with the message hello"
        }
    ]
})

print(response["messages"][-1].content)

Stap 5 — test de agent met alleen navigate_browser

Waarom: we hoeven nu niet alle zeven Playwright-tools tegelijk te gebruiken. Eerst testen we precies de tool die de agent nodig heeft om naar de website te gaan.

Voer uit:

In [ ]:
navigate_agent = create_agent(
    model=llm,
    tools=[navigate_tool]
)

Stap 6 — een synchronische browser aanmaken

Waarom: hiermee geven we navigate_browser precies het type browser dat de synchronische agent verwacht.

Voer eerst alleen dit uit:


Waarom: we moeten de sync_browser buiten de Jupyter-eventloop starten, net zoals we eerder met de Proactor-thread voor de async browser hebben gedaan.

Stap 7 — synchronische Playwright in een aparte thread starten

Waarom: de aparte thread heeft geen Jupyter asyncio-loop, waardoor de Sync API daar wel kan starten.

Voer dit uit:

In [ ]:
import threading
from playwright.sync_api import sync_playwright

sync_playwright_state = {}

def start_sync_playwright():
    p = sync_playwright().start()
    browser = p.chromium.launch(headless=True)

    sync_playwright_state["playwright"] = p
    sync_playwright_state["browser"] = browser

sync_thread = threading.Thread(
    target=start_sync_playwright,
    daemon=True
)

sync_thread.start()
sync_thread.join()

print(sync_playwright_state)

Stap 8 — de Playwright-toolkit opnieuw maken met deze sync_browser.

Waarom: de fout die we kregen was specifiek dat navigate_browser geen synchronische browser had. We geven de toolkit nu de browser die we zojuist hebben gestart.

Voer dit uit:

In [ ]:
sync_toolkit = PlayWrightBrowserToolkit.from_browser(
    sync_browser=sync_playwright_state["browser"]
)

sync_tools = sync_toolkit.get_tools()

print("Aantal tools:", len(sync_tools))

for tool in sync_tools:
    print(tool.name)

Stap 9 — eerst alleen de synchronische navigate_browser testen.

Waarom: voordat we de agent opnieuw maken, controleren we dat de nieuwe toolkit daadwerkelijk met de nieuwe sync_browser kan navigeren. Zo voorkomen we dat we twee problemen tegelijk testen.

Stap 11 — eerst de agent vanuit onze bestaande Proactor-loop uitvoeren.

Waarom: onze playwright_state["loop"] is de loop waarop de async browser draait. We moeten de volledige agent.ainvoke() daar laten uitvoeren in plaats van in de Jupyter-loop.

Voer dit uit:

In [ ]:
agent_future = asyncio.run_coroutine_threadsafe(
    agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": "Navigate to http://eaapp.somee.com/Employee"
            }
        ]
    }),
    playwright_state["loop"]
)

response = agent_future.result(timeout=60)

print(response["messages"][-1].content)

Stap 12 — laat de agent de werknemersnamen ophalen.

Waarom: dit is het eigenlijke doel van les 49. De LLM moet nu zelf bepalen dat hij eerst naar de pagina moet navigeren en daarna get_elements moet gebruiken.

Voer dit uit:

In [ ]:
%pip install lxml

Stap 15 — Controleren of lxml beschikbaar is

Waarom: na de kernelrestart controleren we eerst of Python de nieuwe parser daadwerkelijk kan importeren. Zo voorkomen we dat we de agent opnieuw starten terwijl de dependency nog niet beschikbaar is.

Voer deze cel uit:

In [ ]:
import lxml
print(lxml.__version__)

In [ ]:
agent_future = asyncio.run_coroutine_threadsafe(
    agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Go to http://eaapp.somee.com/Employee "
                    "and extract the names of all employees. "
                    "Return only the employee names."
                )
            }
        ]
    }),
    playwright_state["loop"]
)

response = agent_future.result(timeout=120)

print(response["messages"][-1].content)

Stap 17 — asyncio opnieuw importeren

Waarom: na de kernelrestart is de Python-sessie leeg. Daarom bestaat asyncio nog niet in deze nieuwe kernel.

Voer alleen dit uit:

In [ ]:
import asyncio

Stap 18 — Controleren of de Playwright-browser nog actief is

Waarom: na een kernelrestart zijn ook playwright_state en de bijbehorende Python-objecten verdwenen. We controleren eerst of de oude achtergrondthread nog bestaat voordat we opnieuw iets opstarten.

Voer uit:

In [ ]:
print("playwright_state" in globals())

In [ ]:
import asyncio
import threading
from playwright.async_api import async_playwright

playwright_state = {}
playwright_error = {}

def playwright_worker():
    try:
        loop = asyncio.ProactorEventLoop()
        asyncio.set_event_loop(loop)

        async def start():
            p = await async_playwright().start()
            browser = await p.chromium.launch(headless=True)

            playwright_state["playwright"] = p
            playwright_state["browser"] = browser
            playwright_state["loop"] = loop

        loop.run_until_complete(start())
        loop.run_forever()

    except Exception as e:
        playwright_error["error"] = repr(e)

playwright_thread = threading.Thread(
    target=playwright_worker,
    daemon=True
)

playwright_thread.start()
playwright_thread.join(timeout=10)

print("State:", playwright_state)
print("Error:", playwright_error)

Stap 21 — Playwright Toolkit opnieuw aanmaken

Waarom: na de kernelrestart is ook toolkit verdwenen. We koppelen de actieve browser daarom opnieuw aan de Playwright Browser Toolkit. Daarna kunnen we de agent opnieuw opbouwen.

Voer alleen dit uit:

Stap 22 — De juiste classnaam controleren

Waarom: dit is geen Playwright-probleem. Na de kernelrestart blijkt dat de importnaam die we eerder gebruikten niet beschikbaar is vanuit deze packageversie. We controleren eerst exact welke naam jouw geïnstalleerde langchain-community aanbiedt, voordat we iets aanpassen.

Voer alleen dit uit:

In [ ]:
import langchain_community.agent_toolkits.playwright as pw

print([x for x in dir(pw) if "Play" in x])

Stap 23 — De class rechtstreeks uit de module halen

Waarom: jouw package bevat PlayWRightBrowserToolkit wel, maar exporteert hem blijkbaar niet correct via from ... import .... We kunnen hem daarom rechtstreeks uit de module halen. Dit verandert niets aan de toolkit zelf.

Stap 24 — De class rechtstreeks uit het bronbestand importeren

Waarom: we slaan de foutieve package-export over en importeren de class uit het daadwerkelijke toolkit-modulebestand.

Voer alleen dit uit:

In [ ]:
import inspect
import langchain_community.agent_toolkits.playwright.toolkit as toolkit_module

print([
    name
    for name, obj in inspect.getmembers(toolkit_module, inspect.isclass)
    if "Play" in name or "Browser" in name
])

In [ ]:
import inspect

print(inspect.getsource(toolkit_module)[:5000])

In [ ]:
import runpy

module_globals = runpy.run_path(toolkit_module.__file__)

print("PlayWRightBrowserToolkit" in module_globals)
print([
    name
    for name in module_globals
    if "PlayW" in name
])

In [ ]:
PlayWrightBrowserToolkit = module_globals["PlayWrightBrowserToolkit"]

print(PlayWrightBrowserToolkit)

In [ ]:
toolkit = PlayWrightBrowserToolkit.from_browser(
    async_browser=playwright_state["browser"]
)

tools = toolkit.get_tools()

print("Aantal tools:", len(tools))
print([tool.name for tool in tools])

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=tools
)

print(agent)

Stap 34 — De agent laten navigeren en scrapen

Waarom: de agent is opnieuw aangemaakt. Nu testen we precies het doel van les 49: de LLM krijgt een natuurlijke opdracht en moet zelf de Playwright-tools gebruiken om naar de pagina te gaan en de werknemersnamen op te halen.

Gebruik opnieuw de Proactor-loop waarop Chromium draait:

In [ ]:
print("Loop:", playwright_state["loop"])
print("Loop running:", playwright_state["loop"].is_running())
print("Loop closed:", playwright_state["loop"].is_closed())
print("Browser:", playwright_state["browser"])

In [ ]:
llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=tools
)

agent_future = asyncio.run_coroutine_threadsafe(
    agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": "Navigate to http://eaapp.somee.com/Employee"
            }
        ]
    }),
    playwright_state["loop"]
)

response = agent_future.result(timeout=60)

print(response["messages"][-1].content)

Stap 3 — Laat de agent nu alleen de tabelgegevens ophalen

Waarom: hiermee testen we de tweede Playwright-tool (get_elements) afzonderlijk binnen de agent. Als dit werkt, weten we dat de combinatie van navigeren + tabel uitlezen werkt en kunnen we daarna pas het gemiddelde toevoegen.

In [ ]:
get_elements_tool = next(
    tool for tool in tools
    if tool.name == "get_elements"
)

future = asyncio.run_coroutine_threadsafe(
    get_elements_tool.arun(
        "td",
        ["inner_text"]
    ),
    playwright_state["loop"]
)

result = future.result(timeout=30)

print(result)

Stap 5 — Kijk welke tool-calls de agent probeert te maken

Waarom: bij de timeout krijgen we nu alleen TimeoutError, maar we weten niet of Gemma4 bijvoorbeeld meerdere keren dezelfde tool probeert aan te roepen. We kunnen de agent een kortere vraag geven en na een beperkte tijd bekijken of er al berichten/tool-calls zijn geproduceerd.

In [ ]:
from langchain.tools import tool

@tool
async def get_employee_names() -> str:
    """Reads employee names from the Employee table after navigation has completed."""
    browser = playwright_state["browser"]
    context = browser.contexts[0]
    page = context.pages[0]

    for attempt in range(3):
        try:
            await page.wait_for_load_state("domcontentloaded")
            names = await page.locator("td").all_inner_texts()

            return "\n".join(
                name for name in names
                if name and "Details" not in name and not name.endswith("yrs")
            )
        except Exception as error:
            if "Execution context was destroyed" not in str(error):
                raise
            await asyncio.sleep(1)

    raise RuntimeError("The page kept navigating while employee names were being read.")


agent = create_agent(
    model=llm,
    tools=[navigate_tool, get_employee_names]
)

agent_future = asyncio.run_coroutine_threadsafe(
    agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Navigate to http://eaapp.somee.com/Employee, "
                    "then use get_employee_names. Return only the names."
                )
            }
        ]
    }),
    playwright_state["loop"]
)

try:
    response = agent_future.result(timeout=60)
    print("Agent klaar")
    print(response["messages"][-1].content)
except TimeoutError:
    print("Agent is na 60 seconden nog niet klaar.")
    print("Future running:", not agent_future.done())

Stap 6 — Test alleen de LLM met de Playwright-tools

Waarom: hiermee halen we create_agent tijdelijk uit de vergelijking. We vragen Gemma4 rechtstreeks om een tool-call te maken. Als dat snel gebeurt, zit het probleem in de agent-loop en niet in de LLM of tool-binding.

In [ ]:
llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(
    "Go to http://eaapp.somee.com/Employee and get the employee names."
)

print(response.tool_calls)

Stap 7 — Geef de agent expliciet een korte instructie

Waarom: de agent moet na het navigeren begrijpen dat hij daarna get_elements moet gebruiken. We laten hem niet meteen rekenen; eerst alleen de namen. Dit maakt de agent-loop zo eenvoudig mogelijk.

In [ ]:
agent_future = asyncio.run_coroutine_threadsafe(
    agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Navigate to http://eaapp.somee.com/Employee. "
                    "Then use the get_elements tool with selector 'td' "
                    "and attribute 'inner_text'. "
                    "Return the employee names only."
                )
            }
        ]
    }),
    playwright_state["loop"]
)

try:
    response = agent_future.result(timeout=60)
    print(response["messages"][-1].content)
except TimeoutError:
    print("Timeout")

In [ ]:
print("Future klaar:", agent_future.done())

if agent_future.done():
    print("Resultaat:")
    print(agent_future.result())
else:
    print("Agent draait nog.")

future = asyncio.run_coroutine_threadsafe(
    playwright_state["browser"].contexts[0].pages[0].title(),
    playwright_state["loop"]
)

print("Pagina:", future.result(timeout=10))

Stap 9 — Controleer de pagina direct na de agent-run

Waarom: we willen vaststellen of de <td>-elementen nu daadwerkelijk in de pagina aanwezig zijn. We gebruiken exact dezelfde rechtstreekse tool die eerder succesvol was.

Voer dit uit:

In [ ]:
future = asyncio.run_coroutine_threadsafe(
    get_elements_tool.arun(
        "td",
        ["inner_text"]
    ),
    playwright_state["loop"]
)

result = future.result(timeout=30)

print(result)

Stap 10 — Laat de agent wachten voordat hij de tabel uitleest

Waarom: de cursus verwacht dat de webpagina direct beschikbaar is, maar deze EAApp-pagina heeft kennelijk iets meer tijd nodig voordat de tabel in de DOM staat. We kunnen dit eerst aantonen door de agent expliciet te vertellen om na het navigeren even te wachten voordat hij get_elements gebruikt.

Gebruik deze vraag:

In [ ]:
import time

time.sleep(3)

future = asyncio.run_coroutine_threadsafe(
    get_elements_tool.arun(
        "td",
        ["inner_text"]
    ),
    playwright_state["loop"]
)

result = future.result(timeout=30)

print(result)

In [ ]:
navigate_tool = next(
    tool for tool in tools
    if tool.name == "navigate_browser"
)

print(navigate_tool)

In [ ]:
navigate_future = asyncio.run_coroutine_threadsafe(
    navigate_tool.arun(
        "http://eaapp.somee.com/Employee"
    ),
    playwright_state["loop"]
)

print(navigate_future.result(timeout=30))

time.sleep(3)

In [ ]:
agent_future = asyncio.run_coroutine_threadsafe(
    agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Use the get_elements tool with selector 'td' "
                    "and attribute 'inner_text' to extract the employee table. "
                    "Return the employee names only."
                )
            }
        ]
    }),
    playwright_state["loop"]
)

response = agent_future.result(timeout=120)

print(response["messages"][-1].content)

Stap 15 — Geef de agent de reeds opgehaalde tabeldata

Waarom: hiermee halen we het probleem met de dynamisch geladen webpagina tijdelijk uit de vergelijking. De agent krijgt nog steeds de taak om de gegevens te interpreteren en het gemiddelde te berekenen, maar get_elements hoeft niet opnieuw op een nog niet geladen DOM te draaien.

In [ ]:
table_data = result

agent_future = asyncio.run_coroutine_threadsafe(
    agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Calculate the average employee salary from this table data. "
                    "Return only the average salary.\n\n"
                    f"{table_data}"
                )
            }
        ]
    }),
    playwright_state["loop"]
)

response = agent_future.result(timeout=120)

print(response["messages"][-1].content)

Stap 1 — Begrijp de wijziging

Waarom: LangChain 1.0 heeft de agent-API aangepast. De oude initialize_agent-aanpak blijft volgens de docent werken via de klassieke namespace, maar de nieuwe aanbevolen syntax gebruikt create_agent.

De overgang is conceptueel:

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools
)

Stap 2 — De async wijziging

Waarom: dit is voor onze Playwright-situatie het belangrijkste onderdeel van deze les. De Playwright-tools zijn asynchronous. Bij de nieuwe create_agent gebruik je daarom niet de oude arun()-aanroep van de agent, maar de async variant van invoke.

Bij ons was dat:

In [ ]:
agent_future = asyncio.run_coroutine_threadsafe(
    agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": "Navigate to http://eaapp.somee.com/Employee"
            }
        ]
    }),
    playwright_state["loop"]
)

response = agent_future.result(timeout=60)

Stap 3 — Wat de docent met de twee versies wil aantonen

Waarom: je moet kunnen herkennen welke syntax je in codevoorbeelden tegenkomt.

De klassieke variant gebruikt ongeveer:

In [ ]:
from langchain_classic.agents import initialize_agent

en de nieuwe LangChain 1.x-variant:

In [ ]:
from langchain.agents import create_agent

Ja. Les 50 is inhoudelijk af.

We hebben de kern van de les behandeld:

verschil tussen de klassieke agent-syntax en LangChain 1.x;
initialize_agent versus create_agent;
create_agent uit langchain.agents;
model in plaats van llm;
agent.ainvoke(...) voor de asynchronous agent;
de Playwright async-tools binnen deze nieuwe agentstructuur;
en we hebben deze aanpak daadwerkelijk in jouw omgeving uitgevoerd.

Het enige wat niet exact hetzelfde liep als bij de docent, was de combinatie van get_elements binnen de agent. Dat ligt aan onze Playwright/event-loop-context en niet aan de LangChain 1.0-wijziging die deze les behandelt.

Dus: les 50 → afgerond.

In [ ]:
import os

logs_folder = "logs"
os.makedirs(logs_folder, exist_ok=True)

print(f"Log folder: {os.path.abspath(logs_folder)}")

In [ ]:
log_content = """2026-09-21 10:15:32 INFO Application started
2026-09-21 10:15:35 INFO Connecting to MCP server
2026-09-21 10:15:40 ERROR MCP server connection timeout
2026-09-21 10:15:42 ERROR Failed to show the toast notification
2026-09-21 10:15:45 WARNING NTP server connection slow
2026-09-21 10:15:50 ERROR Permission policy violation
"""

log_file = os.path.join(logs_folder, "main.log")

with open(log_file, "w", encoding="utf-8") as f:
    f.write(log_content)

print(f"Created: {log_file}")

In [ ]:
from langchain_core.tools import tool

In [ ]:
@tool
def summarize_logs() -> str:
    """Read the first 50 lines from each log file and return them."""

    summaries = []

    for filename in os.listdir(logs_folder):
        if filename.endswith(".log"):
            file_path = os.path.join(logs_folder, filename)

            with open(file_path, "r", encoding="utf-8") as f:
                lines = f.readlines()[:50]

            summaries.append(
                f"--- {filename} ---\n" + "".join(lines)
            )

    return "\n\n".join(summaries)

In [ ]:
print(summarize_logs.invoke({}))

Stap 12 — De log-tool aan de agent toevoegen

Waarom: tot nu toe kent onze agent alleen add_numbers. We voegen nu summarize_logs toe, zodat de agent zelf kan bepalen dat hij deze tool moet gebruiken wanneer we naar fouten in de logs vragen.

Maak de agent opnieuw met beide tools:

In [ ]:
from langchain.agents import create_agent

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:latest",
    base_url="http://localhost:11434",
    temperature=0.5
)

In [ ]:
from langchain_core.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

In [ ]:
agent = create_agent(
    model=llm,
    tools=[add_numbers, summarize_logs]
)

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What are the errors in my logs?"
        }
    ]
})

print(result["messages"][-1].content)

In [ ]:
response = llm.invoke("What is 20 + 40?")
print(response.content)

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What are the errors in my logs?"
        }
    ]
})

print(result["messages"][-1].content)

In [ ]:
print([tool.name for tool in [add_numbers, summarize_logs]])

In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

llm = ChatOllama(
    model="qwen2.5:latest",
    base_url="http://localhost:11434",
    temperature=0.5
)

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


@tool
def summarize_logs() -> str:
    """Read the first 50 lines from each log file and return them."""

    summaries = []

    for filename in os.listdir(logs_folder):
        if filename.endswith(".log"):
            file_path = os.path.join(logs_folder, filename)

            with open(file_path, "r", encoding="utf-8") as f:
                lines = f.readlines()[:50]

            summaries.append(
                f"--- {filename} ---\n" + "".join(lines)
            )

    return "\n\n".join(summaries)


agent = create_agent(
    model=llm,
    tools=[add_numbers, summarize_logs]
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What are the errors in my logs?"
        }
    ]
})

for message in result["messages"]:
    print(type(message).__name__)
    print(message)
    print("-" * 80)

In [ ]:
import os
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What are the toast error messages I am getting in the app?"
        }
    ]
})

print(result["messages"][-1].content)

#### And in this section we are going to talk about how we can start building a log parser with the power

of the AI agents and rags and lang chains and local larger language models and things.

### Stap 1 — klaar Je hebt gecontroleerd dat Python jouw bestaande main.log kan lezen.

In [ ]:
from pathlib import Path

log_file = Path(
    r"C:\Users\nihor\agentsforQA\.venv\Section6_PlaywrightToolkit\logs\main.log"
)

def read_log_file():
    return log_file.read_text(encoding="utf-8")

print(read_log_file())

### stap 2: van het lezen van het logbestand een LangChain-tool maken:read_logs als LangChain-tool maken

In [ ]:
from langchain.tools import tool

@tool
def read_logs() -> str:
    """Read the application log file and return its contents."""
    return log_file.read_text(encoding="utf-8")

In [ ]:
print(read_logs.name)
print(read_logs.description)

### Stap 3: controleren of de tool daadwerkelijk de loggegevens kan teruggeven:read_logs.invoke({}) testen

Maak een nieuwe cel en voer alleen dit uit:

In [ ]:
result = read_logs.invoke({})

print(result)

### Stap 4: het LLM aan de tool koppelen.LLM koppelen aan read_logs

Daarbij gebruiken we jouw bestaande lokale Ollama-model. We gaan het LLM eerst laten weten dat read_logs beschikbaar is. Het LLM kan dan later zelf bepalen wanneer het deze tool moet gebruiken.

Voordat we die stap doen, wil ik één ding controleren: welk Ollama-model je in dit notebook gebruikt. Voer in een nieuwe cel uit:

In [ ]:
import ollama

models = ollama.list()

for model in models["models"]:
    print(model["model"])

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0
)

llm_with_tools = llm.bind_tools([read_logs])

print("LLM met tool is klaar.")

### Stap 5: testen of het LLM zelf besluit de tool aan te roepen.

In [ ]:
llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)
llm_with_tools = llm.bind_tools([read_logs])

response = llm_with_tools.invoke(
    "What errors are present in the log file?"
)

print(response)

Perfect, stap 5 is geslaagd. Dit gedeelte van jouw output is het belangrijkste: tool_calls=[{'name': 'read_logs', 'args': {}, ...}]. Dat betekent dat het LLM zelf heeft bepaald: voor deze vraag moet ik de read_logs-tool gebruiken. Maar let op: de tool is nog niet daadwerkelijk uitgevoerd door de agent; het LLM heeft alleen een tool-call voorgesteld. We hebben nu dus: stap 1 Python → main.log, stap 2 read_logs → LangChain tool, stap 3 tool → succesvol uitgevoerd, stap 4 LLM → tool beschikbaar, stap 5 LLM → besluit read_logs te gebruiken. Nu komt het interessante gedeelte.

### Stap 6 — de tool-call daadwerkelijk uitvoeren

Waarom doen we dit? Het LLM heeft gezegd: "name": "read_logs", "args": {}. Wij pakken die instructie eruit en voeren de betreffende tool uit. Daarmee krijgen we: LLM ↓ "Gebruik read_logs" ↓ read_logs.invoke(...) ↓ main.log ↓ loggegevens. Voer alleen deze stap uit. Stuur daarna de output. Daarna gaan we naar de volgende stap: de opgehaalde loggegevens terug naar het LLM sturen zodat het daadwerkelijk een begrijpelijke analyse maakt.

In [ ]:
from pathlib import Path

tool_call = response.tool_calls[0]

log_file = Path.cwd() / "logs" / "main.log"
result = read_logs.invoke(tool_call["args"])

print(result)

Stap 6 is geslaagd. We hebben nu de volledige keten tot aan het ophalen van de data werkend: Gebruiker ↓ LLM ↓ LLM kiest read_logs ↓ read_logs wordt uitgevoerd ↓ main.log ↓ loggegevens. Maar er ontbreekt nog één belangrijk onderdeel: het LLM heeft de opgehaalde logs nog niet teruggekregen. Daardoor kan het ze ook nog niet samenvatten.

### Stap 7 — resultaat terugsturen naar het LLM

Maak een nieuwe codecel:

Waarom doen we dit? We geven het resultaat van read_logs terug aan hetzelfde LLM dat de tool heeft aangeroepen. De volledige flow wordt nu: Vraag ↓ LLM ↓ "Gebruik read_logs" ↓ Tool ↓ main.log ↓ Tool resultaat ↓ LLM ↓ begrijpelijk antwoord. Dit is belangrijk omdat je hiermee het principe uit de cursus daadwerkelijk hebt uitgevoerd: LLM → tool → externe data → LLM → antwoord.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

tool_message = ToolMessage(
    content=result,
    tool_call_id=tool_call["id"]
)

final_response = llm_with_tools.invoke([
    HumanMessage(content="What errors are present in the log file?"),
    response,
    tool_message
])

print(final_response.content)

Ja. Dit is precies het resultaat dat we wilden bereiken. Met qwen3:8b ging het veel sneller en het model heeft de logs daadwerkelijk geïnterpreteerd. We hebben nu de volledige basisworkflow werkend: Gebruiker ↓ qwen3:8b ↓ besluit: read_logs gebruiken ↓ read_logs tool ↓ main.log ↓ loggegevens ↓ qwen3:8b ↓ samenvatting. En je hebt hiermee het belangrijkste principe uit de les zelf gebouwd. Tot nu toe hebben we: Stap 1: main.log lezen, Stap 2: read_logs als LangChain-tool maken, Stap 3: de tool uitvoeren, Stap 4: LLM + tool verbinden, Stap 5: LLM laten besluiten de tool te gebruiken, Stap 6: de tool-call uitvoeren, Stap 7: resultaat teruggeven aan het LLM → werkend.

De volgende stap is interessant: we gaan het handmatige gedeelte dat we nu zelf deden automatiseren met een agent. Dat is precies de overgang van: "LLM met een tool" naar: "AI-agent die zelfstandig tools gebruikt." We doen dat als Stap 8, en gebruiken daarbij je snelle qwen3:8b.

In [ ]:
tool_call = response.tool_calls[0]
result = read_logs.invoke(tool_call["args"])
tool_message = ToolMessage(
    content=result,
    tool_call_id=tool_call["id"]
)

Hier is een voorbeeld met de flows achter elkaar, met pijlen in plaats van verticale stapels:

**Agentcyclus (handmatig nagebouwd):**
Gebruiker stelt vraag -> LLM bepaalt welke tool nodig is -> LLM geeft tool-call -> Tool wordt uitgevoerd -> Tool geeft data terug -> Data gaat terug naar LLM -> LLM maakt antwoord

**Zonder agent (handmatig):**
```python
tool_call = response.tool_calls[0]
result = read_logs.invoke(tool_call["args"])
```

**Met een agent (conceptueel):**
Gebruiker -> Agent -> LLM -> "Ik heb read_logs nodig" -> Agent voert read_logs uit -> resultaat -> LLM -> antwoord

**Voorbeeld in de praktijk:**
Gebruiker vraagt "Wat staat er in het logbestand?" -> Agent -> LLM -> "Ik heb read_logs nodig" -> Agent voert read_logs uit -> main.log wordt gelezen -> resultaat -> LLM -> "Er staan 3 fouten in het logbestand: ..."

### Stap 8: de agent bouwen, waarbij we jouw werkende qwen3:8b gebruiken.Tot nu toe moesten wij zelf de tool-call uitvoeren. Nu laten we een agent dat automatisch doen.

Waarom deze stap? We geven de agent twee dingen:
LLM + read_logs tool -> Agent
De agent kan vervolgens zelf bepalen: welke tool nodig is; de tool aanroepen; het resultaat verwerken; eventueel opnieuw een tool gebruiken; uiteindelijk een antwoord geven. Dit is dus anders dan onze vorige handmatige aanpak. Als je krijgt: Agent is klaar.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[read_logs]
)

print("Agent is klaar.")

### Stap 9 — de agent een vraag geven
Nu gaan we testen of de agent daadwerkelijk zelfstandig de tool gebruikt.

Hier doen we iets belangrijks: we geven de agent alleen de vraag. We zeggen dus niet dat hij read_logs moet gebruiken. De agent moet zelf ontdekken dat hij daarvoor de tool nodig heeft. De gewenste flow is:
"What errors are present?" -> Agent -> qwen3:8b -> "I need read_logs" -> read_logs -> main.log -> qwen3:8b -> antwoord
Dit is het eerste moment waarop we echt kunnen zeggen dat we een AI-agent hebben in plaats van alleen een LLM waaraan we handmatig een tool-call hebben gekoppeld. Voer deze cel uit. Stuur daarna het antwoord dat je krijgt.

In [ ]:
from pathlib import Path
from langchain_core.tools import tool

# Ensure the log file is defined in this notebook session
logs_dir = Path.cwd() / "logs"
logs_dir.mkdir(exist_ok=True)
log_file = logs_dir / "main.log"

if not log_file.exists():
    log_file.write_text(
        "2026-09-21 10:15:32 INFO Application started\n"
        "2026-09-21 10:15:35 INFO Connecting to MCP server\n"
        "2026-09-21 10:15:40 ERROR MCP server connection timeout\n"
        "2026-09-21 10:15:42 ERROR Failed to show the toast notification\n"
        "2026-09-21 10:15:45 WARNING NTP server connection slow\n"
        "2026-09-21 10:15:50 ERROR Permission policy violation\n",
        encoding="utf-8"
    )

# Rebuild the tool so it uses the actual log_file in this session

@tool
def read_logs() -> str:
    """Read the application log file and return its contents."""
    return log_file.read_text(encoding="utf-8")

agent = create_agent(
    model=llm,
    tools=[read_logs]
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What errors are present in the log file?"
        }
    ]
})

print(result["messages"][-1].content)

print(result["messages"][-1].content)

Goed. Stap 9 is geslaagd. Dit is een belangrijk verschil met wat we hiervoor deden. Jij gaf de agent alleen: "What errors are present in the log file?" De agent heeft vervolgens zelf de read_logs-tool gebruikt en het resultaat verwerkt. We hebben nu: Stap 1 → logbestand lezen, Stap 2 → LangChain-tool maken, Stap 3 → tool testen, Stap 4 → LLM koppelen, Stap 5 → tool calling testen, Stap 6 → tool-call uitvoeren, Stap 7 → resultaat terug naar LLM, Stap 8 → agent bouwen, Stap 9 → agent zelfstandig testen.

### Stap 10 — iets interessanters testen 
Nu gaan we kijken of de agent niet alleen de logs kan reproduceren, maar ze ook kan analyseren.

Hier testen we een belangrijk verschil:
"What errors are present?" -> opsommen
versus:
"Analyze the log" -> informatie uit logs halen -> volgorde begrijpen -> problemen interpreteren
Dit komt dichter bij het doel van de cursus: een QA-engineer moet uiteindelijk niet alleen weten welke errors er staan, maar ook vragen kunnen stellen over de betekenis en context ervan. Voer deze uit en stuur het antwoord. Daarna kunnen we bekijken of de agent correct redeneert over jouw logbestand.

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Analyze the log file. What are the main problems and what happened first?"
        }
    ]
})

print(result["messages"][-1].content)

Dit werkt, maar er zit meteen een belangrijke QA-les in het resultaat. De agent heeft de log correct gelezen en de gebeurtenissen chronologisch geplaatst. Maar hij maakt ook een ongefundeerde causaliteitsclaim: "This suggests the MCP server failure was the root cause of multiple problems." Dat staat niet in de log. We weten alleen dat de MCP-timeout om 10:15:40 gebeurde en de toast-error twee seconden later kwam. We weten niet dat de timeout de oorzaak was. Dat is precies het soort hallucination/inference waar je als QA-engineer alert op moet zijn. We kunnen dit later oplossen met een betere prompt, bijvoorbeeld: Only make claims that are directly supported by the log. Clearly distinguish observed facts from hypotheses. Do not infer causality unless the log provides evidence for it.

### Stap 11 — wat we nu hebben gebouwd

Onze huidige architectuur is:

text
                ┌──────────────┐
User ──────────►│  AI Agent    │
                └──────┬───────┘
                       ↓
                 qwen3:8b
                       ↓
                ┌──────────────┐
                │  read_logs   │
                └──────┬───────┘
                       ↓
                   main.log
                       ↓
                 loggegevens
                       ↓
                  qwen3:8b
                       ↓
                  antwoord
Dit is de agent-versie zonder RAG. De volgende stap uit de cursus is het probleem dat de docent beschreef: Wat gebeurt er als je niet één klein main.log hebt, maar duizenden of voortdurend veranderende logbestanden? Dan wil je niet iedere keer: Agent → alle bestanden openen → alles lezen → analyseren. Daarom gaan we nu naar RAG. En hier kunnen we jouw bestaande nomic-embed-text:latest gebruiken.

### Stap 12 — eerst RAG los testen

We gaan nog geen bestaande qa_db gebruiken. Eerst maken we specifiek voor deze logparser een kleine vector database, zodat je precies ziet wat de cursus bedoelt.

Waarom? qwen3:8b is ons chat/agent-model. nomic-embed-text is ons embedding-model. De rollen zijn dus verschillend:
qwen3:8b -> begrijpen / beslissen / antwoorden
nomic-embed-text -> tekst → vector
Voer deze cel uit. Als je krijgt: Embeddings model klaar. gaan we daarna de main.log in stukken opdelen en in een vector database zetten.

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest"
)

print("Embeddings model klaar.")

### Stap 13: de log opdelen in chunks.

Waarom doen we dat? Een vector database werkt niet optimaal als we een groot logbestand als één enorm tekstblok opslaan. We willen kleinere stukken tekst kunnen terugvinden die relevant zijn voor een vraag.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

log_text = read_logs.invoke({})

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.create_documents([log_text])

print("Aantal chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk.page_content)

Je logbestand is klein, dus waarschijnlijk krijg je maar één of enkele chunks. De workflow wordt nu: main.log -> log_text -> Text Splitter -> chunks -> embedding -> vector database. We doen dit bewust apart van de agent. Zo kunnen we straks precies zien wat RAG toevoegt. Voer deze cel uit en stuur me de output van de chunks.

Goed. Stap 13 is geslaagd. Je log is nu opgesplitst in 2 chunks: Chunk 1: 10:15:32 → 10:15:45, INFO + MCP connection + toast error + NTP warning. Chunk 2: 10:15:50, Permission policy violation. Dit is precies wat we nodig hebben voordat we embeddings maken.

### Stap 14 — embeddings maken en opslaan in Chroma
Nu zetten we de twee chunks om naar vectoren en slaan we ze op in een lokale Chroma-database.

In [ ]:
%pip install -U langchain-chroma chromadb

from langchain_chroma import Chroma
import ollama

installed_models = [m["model"] for m in ollama.list()["models"]]

if "nomic-embed-text:latest" not in installed_models:
    print("Pulling nomic-embed-text:latest...")
    ollama.pull("nomic-embed-text:latest")

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="log_parser",
    persist_directory="./log_parser_db"
)

print("Chunks opgeslagen in Chroma.")

Goed. Stap 14 is geslaagd. De installatie is ook gelukt. De WARNING-meldingen over PATH zijn voor deze notebook niet het probleem. Belangrijk is: Successfully installed ... langchain-chroma-1.1.0 chromadb-1.5.9 ... Chunks opgeslagen in Chroma. Je hebt nu dus daadwerkelijk een lokale vector database: ./log_parser_db. En de workflow is nu: main.log -> 2 chunks -> nomic-embed-text -> embeddings -> Chroma -> log_parser_db.

### Stap 15 — RAG retrieval testen

Voordat we RAG aan de agent koppelen, testen we eerst of Chroma daadwerkelijk de juiste chunk kan terugvinden.

Waarom doen we dit? We vragen nu niet aan het LLM om het antwoord te geven. We vragen rechtstreeks aan de vector database: Welke opgeslagen tekst is het meest relevant voor mijn vraag? Als RAG goed werkt, zou je vooral Chunk 1 terug moeten krijgen, omdat daarin staat: Connecting to MCP server, MCP server connection timeout. Dit is een belangrijke test: Vraag -> Embedding van de vraag -> Chroma zoekt vergelijkbare vector -> relevante chunk. Voer deze cel uit en stuur de output. Daarna bouwen we de RAG-tool die de agent kan gebruiken.

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 1}
)

docs = retriever.invoke(
    "What happened with the MCP server?"
)

for doc in docs:
    print(doc.page_content)

Dit is de output van stap 15. De retriever heeft inderdaad de relevante chunk uit Chroma teruggevonden. Dat betekent dat de basis van onze RAG werkt: vraag -> retriever -> relevante log-chunk.

### stap 16: de retriever als LangChain-tool beschikbaar maken voor de agent.

Waarom: tot nu toe kunnen we de logs rechtstreeks lezen én kunnen we met Chroma relevante stukken ophalen. Nu geven we die zoekfunctie aan de agent, zodat de agent zelf kan bepalen wanneer hij de loggegevens moet ophalen.

In [ ]:
from langchain.tools import tool

@tool
def search_logs(question: str) -> str:
    """Search the application logs for information relevant to a question."""
    
    docs = retriever.invoke(question)

    return "\n\n".join(
        doc.page_content for doc in docs
    )

Test daarna eerst alleen de tool:

In [ ]:
result = search_logs.invoke({
    "question": "What happened with the MCP server?"
})

print(result)

Goed. Stap 16 is geslaagd. De search_logs-tool haalt via Chroma de relevante loggegevens op.

### stap 17: de RAG-tool aan de agent geven.
Waarom: nu kan de agent niet alleen een bestand lezen, maar ook gericht zoeken in de opgeslagen log-chunks.

In [ ]:
agent_rag = create_agent(
    model=llm,
    tools=[search_logs]
)

print("RAG-agent is klaar.")

De agent zou nu eerst search_logs moeten gebruiken en daarna op basis van de gevonden logregels antwoord geven.

### 17b: de agent daadwerkelijk testen. 
Dit is nodig om te controleren of hij zelfstandig search_logs aanroept en het opgehaalde fragment gebruikt.

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What happened with the MCP server?"
        }
    ]
})

print(result["messages"][-1].content)

Dit werkt technisch goed. Stap 17b is geslaagd. De agent heeft de RAG-tool gebruikt en een antwoord gegenereerd.

Er zit wel één inhoudelijk probleem in het antwoord: de agent zegt dat de MCP-timeout waarschijnlijk de toast notification error veroorzaakte. Dat staat niet in de log. Ook de conclusie dat de NTP-waarschuwing "unrelated" is, is niet bewezen.

Dit is juist een nuttige observatie voor je QA/agent-opzet: RAG haalt de juiste informatie op, maar de LLM kan daar nog steeds ongefundeerde conclusies aan toevoegen.

### 18: de agent dwingen onderscheid te maken tussen feiten en aannames.

In [ ]:
agent_rag = create_agent(
    model=llm,
    tools=[search_logs],
    system_prompt="""
You are a log analysis assistant.

Only make claims that are directly supported by the log data.
Do not assume that one log event caused another event unless the logs explicitly provide evidence of causality.
Clearly distinguish observed facts from hypotheses.
If the logs do not provide enough information, say so.
"""
)

print("RAG-agent met strikte logregels is klaar.")

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What happened with the MCP server?"
        }
    ]
})

print(result["messages"][-1].content)

In [ ]:
agent_rag = create_agent(
    model=llm,
    tools=[search_logs],
    system_prompt="""
You are a log analysis assistant.

Only report facts that are directly present in the retrieved logs.

Do NOT infer causality from timestamps or event order.
Do NOT say that one event caused, triggered, resulted in, or was a consequence of another event unless the log explicitly states this relationship.

Do NOT use speculative language such as:
- likely
- probably
- may have caused
- could be due to
- consequence of

If causality is not explicitly documented, state only the individual events.

Clearly distinguish observed facts from hypotheses.
If the logs do not provide enough information, say so.
"""
)

print("Strikte RAG-agent klaar.")

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What happened with the MCP server?"
        }
    ]
})

print(result["messages"][-1].content)

Stap 18 is nu geslaagd. De agent doet nu wat we willen: hij haalt de relevante informatie uit Chroma, hij rapporteert de gebeurtenissen als feiten, hij verzint geen causale relatie, hij geeft expliciet aan wanneer de logs onvoldoende bewijs leveren. We hebben daarmee de volledige keten werkend: Gebruiker → Agent → RAG-tool → Chroma → relevante log-chunk → LLM → antwoord. 

### De volgende stap is stap 19: testen met een andere vraag. 
Daarmee controleren we of de Daarmee controleren we of de agent niet alleen goed werkt voor de MCP-server, maar ook andere informatie uit de logs kan terugvinden.

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What errors are present in the logs?"
        }
    ]
})

print(result["messages"][-1].content)

Hier zien we een belangrijk probleem met onze huidige RAG-configuratie. De agent zegt dat alleen de Permission policy violation een error is. Maar de logs bevatten drie ERROR-regels: MCP server connection timeout, Failed to show the toast notification, Permission policy violation. De oorzaak is dat we in stap 15 de retriever hebben ingesteld op: search_kwargs={"k": 1}. Daardoor haalt Chroma maar één chunk op. De vraag "What errors are present in the logs?" vereist informatie uit beide chunks. Dit is dus geen LLM-probleem, maar een retrieval-configuratieprobleem.

### Stap 19b — meer dan één chunk ophalen

Pas de retriever aan naar k=2:Waarom: we hebben momenteel twee chunks in Chroma. Met k=2 krijgt de agent beide relevante chunks en kan hij alle errors zien.

Daarna hoeven we de search_logs-tool niet opnieuw te maken, want die gebruikt automatisch deze retriever.



In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("Retriever aangepast naar k=2.")

Test opnieuw:

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What errors are present in the logs?"
        }
    ]
})

print(result["messages"][-1].content)

Goed. Stap 19b is geslaagd.

Nu zien we dat de RAG-agent alle drie de ERROR-meldingen kan terugvinden. De volgorde is alleen niet chronologisch, omdat Chroma resultaten rangschikt op relevantie voor de vraag.

Dat is op zichzelf geen probleem, maar voor loganalyse is chronologische volgorde vaak belangrijk.

### 20: testen of de agent ook een tijdlijn correct kan reconstrueren.
Waarom deze stap: hiermee testen we of de combinatie RAG + LLM niet alleen de juiste informatie vindt, maar die informatie ook correct kan ordenen op basis van timestamps.

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "List all events in chronological order, from earliest to latest. Only use information explicitly present in the logs."
        }
    ]
})

print(result["messages"][-1].content)

Dit is correct. Stap 20 is geslaagd. De agent kan nu: de logs via RAG ophalen, relevante informatie terugvinden, meerdere chunks combineren, errors identificeren, gebeurtenissen chronologisch ordenen, causaliteit niet als feit presenteren wanneer die niet in de logs staat. Daarmee hebben we de kern van de log-parser met RAG gebouwd. De architectuur is nu: Gebruiker -> LangChain Agent -> search_logs tool -> Chroma Vector DB -> relevante log-chunks -> Qwen3:8b -> antwoord. Er is nog één belangrijk onderdeel uit de les dat we nog moeten bouwen: de automatische verwerking van nieuwe/gewijzigde logbestanden. Op dit moment hebben we de vector database één keer gevuld. Als main.log morgen nieuwe regels krijgt, staan die niet automatisch in Chroma. Dat is waarschijnlijk de volgende stap in de cursus: zorgen dat de RAG-database wordt bijgewerkt wanneer nieuwe logs binnenkomen. Als we de les stap voor stap blijven volgen, gaan we daar nu mee verder.

Logbestand lezen, Logfunctie als LangChain-tool maken, Tool uitvoeren, LLM aan de tool koppelen, Tool calling testen, Toolresultaat teruggeven aan de LLM, Agent bouwen, Agent testen, RAG-opzet maken, Logs chunken, Chroma-vector database maken, Retrieval testen, RAG-tool maken, RAG-agent bouwen, RAG-agent testen, Retrieval verbeteren van k=1 naar k=2, Chronologische analyse testen, Hallucinaties/ongefundeerde causaliteit met een striktere prompt beperken.

Ja. Dit fragment is de inhoud van les 54 en het sluit direct aan op wat wij in les 53 hebben gebouwd. De kern van deze les is niet meer het bouwen van RAG, maar het laten zien hoe een agent zelf bepaalt welke tool hij moet gebruiken om logvragen te beantwoorden. In de cursus gebeurt dit ongeveer zo: Gebruiker -> Agent -> bepaalt welke tool nodig is -> summarize_logs -> logbestanden -> LLM -> samenvatting / antwoord. Voor jouw situatie hebben we al een iets uitgebreidere variant: Gebruiker -> RAG Agent -> search_logs -> Chroma -> relevante log-chunks -> Qwen3:8b -> antwoord. Het belangrijke verschil is dat de cursus een summarize_logs-tool gebruikt die de eerste regels van logbestanden leest en samenvat. Wij hebben inmiddels een RAG-gebaseerde search_logs-tool, waardoor de agent gericht relevante delen van de logs kan ophalen. Ook de waarschuwing uit de les over grote hoeveelheden logdata hebben we eigenlijk al ondervangen. In plaats van honderden regels rechtstreeks naar het lokale LLM te sturen, gebruiken wij embeddings + Chroma om eerst relevante chunks te selecteren. Dus voor les 54 hoeven we niet opnieuw de code uit de video letterlijk na te bouwen. We kunnen de concepten aan onze bestaande implementatie koppelen. De belangrijkste dingen die de docent hier demonstreert zijn: de agent krijgt een natuurlijke vraag, bijvoorbeeld "What are the errors in my logs?", de agent herkent dat hij daarvoor de log-tool nodig heeft, de tool haalt de loggegevens op, de LLM analyseert die gegevens, de agent geeft een antwoord terug. Dezelfde agent kan daarna een andere logvraag krijgen, bijvoorbeeld "What are the toast error messages?", de agent gebruikt opnieuw de log-tool, zonder dat de gebruiker zelf hoeft te bepalen welke functie aangeroepen moet worden. Dat laatste is eigenlijk het belangrijkste punt van deze les: tool selection door de agent. En dat hebben wij al daadwerkelijk getest. Onze agent heeft bijvoorbeeld zelfstandig search_logs gebruikt voor: "What happened with the MCP server?" en: "What errors are present in the logs?" en: "List all events in chronological order...". Daarmee hebben we het centrale doel van deze les al praktisch gerealiseerd.

### Stap 1 — controleren welke tool de agent gebruikt

We testen of de agent zelfstandig begrijpt dat hij de search_logs-tool moet gebruiken om de vraag te beantwoorden.

Waarom deze stap: hiermee controleren we het belangrijkste concept van les 54: de gebruiker stelt alleen een vraag en de agent bepaalt zelf welke tool nodig is.

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What errors are present in the logs?"
        }
    ]
})

print(result["messages"][-1].content)

### Stap 2 — controleren of de agent verschillende logvragen aankan

De docent vraagt daarna een specifiekere vraag over de logs. Wij doen hetzelfde met de toast errors.

Waarom deze stap: hiermee testen we of de agent niet alleen algemene errors kan vinden, maar via dezelfde tool ook gericht relevante informatie uit de logs kan ophalen.

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What toast error messages are present in the logs?"
        }
    ]
})

print(result["messages"][-1].content)

### Stap 3 — controleren of de agent de logs kan samenvatten

Dit sluit direct aan op wat de docent in les 54 doet met summarize_logs.

Waarom deze stap: we testen nu of de agent niet alleen losse fouten kan vinden, maar meerdere logevents kan combineren tot één begrijpelijke samenvatting.

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Summarize the important events in the logs. Only use information explicitly present in the logs."
        }
    ]
})

print(result["messages"][-1].content)

### Stap 4 — controleren of de agent een analysevraag kan beantwoorden

Nu gaan we iets dichter naar het voorbeeld uit de les: niet alleen vragen welke errors er zijn, maar de agent vragen om de belangrijkste problemen te identificeren.

Waarom deze stap: hiermee testen we of de agent de opgehaalde loggegevens kan interpreteren en structureren, terwijl hij zich aan onze regel houdt om alleen conclusies te trekken die door de logs worden ondersteund.

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What are the main problems shown in the logs? Only report what is directly supported by the logs."
        }
    ]
})

print(result["messages"][-1].content)

### Stap 5 — controleren of de agent meerdere soorten vragen kan afhandelen
In de video laat de docent zien dat dezelfde agent opnieuw een vraag krijgt en opnieuw de juiste logtool gebruikt. We testen dat principe met een tijdsvraag.

Waarom deze stap: hiermee controleren we of de agent niet afhankelijk is van één specifiek vraagtype, maar zelfstandig relevante informatie uit de logs kan ophalen en gebruiken.


In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What happened first in the logs, and what happened last?"
        }
    ]
})

print(result["messages"][-1].content)

### Stap 5b — controleren wat de retriever daadwerkelijk teruggeeft

We gaan eerst de agent even buiten beschouwing laten en rechtstreeks kijken welke documenten Chroma voor deze vraag teruggeeft.

Waarom deze stap: hiermee bepalen we of het probleem bij de retrieval zit of bij het LLM. Als Chroma de juiste logregels teruggeeft, weten we dat de hallucination tijdens het genereren van het antwoord ontstaat.

In [ ]:
docs = retriever.invoke(
    "What happened first in the logs, and what happened last?"
)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Retrieved document {i} ---")
    print(doc.page_content)

### Stap 5c — agent dwingen alleen de opgehaalde logs te gebruiken

We passen de system_prompt van onze agent verder aan. De agent moet niet alleen weten dat hij geen causaliteit mag verzinnen, maar ook dat hij nooit informatie uit zijn eigen kennis mag gebruiken wanneer die niet in de opgehaalde logs staat.

Waarom deze stap: de vorige test liet zien dat de retriever correcte data levert, maar het LLM toch informatie uit een andere context kan produceren. We willen de grens tussen "retrieved evidence" en "model knowledge" expliciet maken.

In [ ]:
agent_rag = create_agent(
    model=llm,
    tools=[search_logs],
    system_prompt="""
You are a log analysis assistant.

IMPORTANT:
You must answer questions ONLY using information returned by the search_logs tool.

Never use information from your own knowledge or memory.
Never invent log entries, timestamps, users, events, errors, or system messages.

If the retrieved logs do not contain enough information to answer the question, say:
"The retrieved logs do not contain enough information to answer this question."

Only report facts explicitly present in the retrieved logs.

Do NOT infer causality from timestamps or event order.
Do NOT say that one event caused, triggered, resulted in, or was a consequence of another event unless the logs explicitly state this relationship.

Do NOT use speculative language such as:
- likely
- probably
- may have caused
- could be due to
- consequence of

When asked about chronological order, use the timestamps from the retrieved logs.
"""
)

print("Strikte RAG-agent klaar.")

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What happened first in the logs, and what happened last?"
        }
    ]
})

print(result["messages"][-1].content)

Goed. Stap 5c is geslaagd. De agent gebruikt nu daadwerkelijk de informatie uit de opgehaalde logs en verzint geen gebeurtenissen meer. Daarmee hebben we in les 54 nu getest: Algemene error-vraag → correct. Specifieke toast-vraag → correct. Samenvatting → correct. Analyse van problemen → correct. Chronologische vraag → eerst fout, daarna gecorrigeerd met een strengere prompt. Retriever rechtstreeks gecontroleerd → correcte data. Agent opnieuw getest → correcte data. Dit laatste is belangrijk voor je QA-perspectief: we hebben niet alleen gekeken of het eindresultaat goed was, maar ook vastgesteld waar de fout zat. De retriever leverde correcte data; het LLM hallucineerde. Daarna hebben we de agent-instructies aangepast en opnieuw getest. We kunnen nu verder met het volgende onderdeel van les 54.

### Stap 6 — de agent een complexere logvraag geven

We gaan nu een vraag stellen waarin de agent meerdere stukken informatie uit de logs moet combineren.

Waarom deze stap: hiermee testen we of de agent niet alleen één logregel kan terugvinden, maar meerdere gebeurtenissen kan combineren zonder informatie te verzinnen.

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Summarize all errors in the logs, including their timestamps. Only use information returned by the logs."
        }
    ]
})

print(result["messages"][-1].content)

### Stap 7 — testen met een vraag die geen direct antwoord in de logs heeft

Nu testen we een belangrijk onderdeel van een betrouwbare log-agent: wat doet hij wanneer de gevraagde informatie niet in de logs staat?

Waarom deze stap: een goede agent moet niet proberen een antwoord te verzinnen wanneer het bewijs ontbreekt. Hij moet aangeven dat de logs onvoldoende informatie bevatten.

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Which user experienced the MCP server connection timeout? Only use information returned by the logs."
        }
    ]
})

print(result["messages"][-1].content)

#### Stap 8 — laatste controle van de agent

We doen nog één test die sterk lijkt op het voorbeeld uit de les: een natuurlijke vraag over een specifiek probleem.

Waarom deze stap: hiermee controleren we of de agent in een realistische situatie zelfstandig de juiste informatie uit de logs haalt en daar een bruikbaar antwoord van maakt.

In [ ]:
result = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What problems are related to the MCP server? Only use information explicitly present in the logs."
        }
    ]
})

print(result["messages"][-1].content)

Goed. Stap 8 is geslaagd. De agent heeft alleen de MCP-serverfout genoemd die daadwerkelijk in de logs staat en geen andere problemen aan de MCP-server gekoppeld. Daarmee hebben we les 54 praktisch doorlopen. De belangrijkste concepten uit de les zijn bij jou werkend getest: de agent kiest zelfstandig de juiste tool, de tool haalt de relevante loggegevens op, de agent kan algemene en specifieke logvragen beantwoorden, meerdere logchunks kunnen worden gecombineerd, de agent kan chronologische informatie reconstrueren, ontbrekende informatie wordt niet ingevuld met verzonnen gegevens, we hebben een hallucination gevonden en vervolgens de agentprompt aangescherpt, de uiteindelijke antwoorden zijn gecontroleerd tegen de daadwerkelijke logdata. Les 54 is hiermee klaar. Je kunt nu door naar les 55.

Dit is het volgende stuk van de les. Hier introduceert de docent het opslaan van de logdata in een vector database en het controleren of die database al bestaat, zodat we niet iedere keer opnieuw embeddings hoeven te maken.

Wij hebben een groot deel hiervan al gedaan in onze vorige stappen. Onze log_parser_db bestaat namelijk al en bevat de twee chunks van main.log.

We gaan daarom niet alles opnieuw bouwen. We volgen het principe van de les en maken eerst de controle op de bestaande database.

Stap 1 — controleren of onze vector database al bestaat

Waarom deze stap: embeddings berekenen is relatief kostbaar. Als log_parser_db al bestaat, willen we niet iedere keer opnieuw dezelfde logdata embedden.

In [ ]:
from pathlib import Path

db_path = Path("./log_parser_db")

if db_path.exists():
    print("Vector database bestaat al.")
else:
    print("Vector database bestaat nog niet.")

Goed. Dat bevestigt dat onze bestaande log_parser_db wordt gevonden.

Stap 2 — alleen embedden als de database nog niet bestaat

Nu gaan we precies het principe uit de les toepassen: bestaat de database al, dan slaan we het dure embeddingproces over.

Waarom deze stap: zonder deze controle zou je bij iedere uitvoering opnieuw embeddings maken voor dezelfde logs. Dat kost onnodig tijd en rekenkracht.

Maak een nieuwe codecel en voer uit:

In [ ]:
if not db_path.exists():
    print("Vector database bestaat nog niet. Embeddings worden gemaakt.")

    log_text = read_log_file()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50
    )

    chunks = splitter.create_documents([log_text])

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="log_parser",
        persist_directory="./log_parser_db"
    )

    print("Logs gechunked en opgeslagen in Chroma.")

else:
    print("Vector database bestaat al. Embedding wordt overgeslagen.")

Goed. Stap 2 is geslaagd.

We hebben nu precies de optimalisatie uit de les geïmplementeerd: als log_parser_db al bestaat, worden de logs niet opnieuw gechunked en geëmbed.

Stap 3 — de bestaande database opnieuw gebruiken

Nu moeten we ervoor zorgen dat we bij een bestaande database gewoon opnieuw een vectorstore kunnen gebruiken.

Waarom deze stap: in een nieuwe uitvoering van het notebook kan vectorstore nog niet als Python-variable bestaan, terwijl de databasebestanden wel op schijf staan. We willen de bestaande Chroma-database dan kunnen openen zonder opnieuw embeddings te maken.

Maak een nieuwe codecel:

In [ ]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

db_path = Path("./log_parser_db")

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest"
)

if db_path.exists():
    vectorstore = Chroma(
        collection_name="log_parser",
        embedding_function=embeddings,
        persist_directory="./log_parser_db"
    )

    print("Bestaande Chroma database geladen.")
else:
    print("Vector database bestaat niet.")

Goed. De bestaande Chroma-database is nu succesvol geladen. We gaan nu controleren of de opgeslagen logchunks daadwerkelijk uit de database kunnen worden opgehaald.

Stap 2 — Test de retrieval uit de bestaande database

We vragen de vector database naar informatie over de MCP-server. Hiermee testen we dat we de opgeslagen embeddings kunnen gebruiken zonder de logs opnieuw te embedden.

Waarom deze stap: dit is precies het doel van de optimalisatie uit deze les: bestaande embeddings hergebruiken in plaats van iedere keer opnieuw embeddings te maken.

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

docs = retriever.invoke("What happened with the MCP server?")

for doc in docs:
    print(doc.page_content)

Goed. De retrieval werkt correct: de bestaande Chroma-database levert de opgeslagen loggegevens terug. Er is dus geen nieuwe embedding uitgevoerd.

Stap 3 — De bestaande search_logs-tool opnieuw koppelen

Nu maken we de retriever weer beschikbaar via dezelfde tool die we eerder voor de agent gebruikten.

Waarom deze stap: de agent moet niet rechtstreeks de vector database aanspreken. De agent gebruikt de search_logs-tool, en die tool haalt de relevante informatie uit de bestaande Chroma-database.

In [ ]:
from langchain.tools import tool

@tool
def search_logs(question: str) -> str:
    """Search the stored application logs for information relevant to a question."""
    docs = retriever.invoke(question)
    return "\n\n".join(doc.page_content for doc in docs)

Test daarna alleen de tool:

In [ ]:
result = search_logs.invoke({
    "question": "What errors occurred?"
})

print(result)

Dit werkt. De search_logs-tool haalt de informatie nu uit de bestaande Chroma-database.

De volgorde is alleen interessant: Chroma geeft de meest semantisch relevante chunks terug, niet noodzakelijk de chronologische volgorde. Dat is normaal voor vector retrieval.

Stap 4 — De RAG-agent opnieuw aansluiten

Nu koppelen we deze geoptimaliseerde search_logs-tool weer aan de agent.

Waarom deze stap: hiermee is de keten compleet: vraag → agent → search_logs → bestaande Chroma-database → relevante loggegevens → antwoord.

In [ ]:

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

print("LLM geladen.")

Stap 4b — search_logs opnieuw aanmaken

Waarom deze stap: de agent kan pas worden aangemaakt als de tool waarnaar hij verwijst (search_logs) weer in het geheugen van de kernel staat.

In [ ]:
from langchain.tools import tool

@tool
def search_logs(question: str) -> str:
    """Search the stored application logs for information relevant to a question."""
    docs = retriever.invoke(question)
    return "\n\n".join(doc.page_content for doc in docs)

print("search_logs tool geladen.")

Goed. Nu zijn zowel llm als search_logs beschikbaar.

Stap 4c — RAG-agent opnieuw aanmaken

Waarom deze stap: we koppelen het taalmodel aan de search_logs-tool, zodat de agent vragen kan beantwoorden op basis van de bestaande Chroma-database.

In [ ]:
from langchain.agents import create_agent

agent_rag = create_agent(
    model=llm,
    tools=[search_logs],
    system_prompt="""
You are a log analysis assistant.

Answer questions ONLY using information returned by the search_logs tool.

Never invent log entries, timestamps, users, events, errors, or system messages.

If the retrieved logs do not contain enough information to answer the question, say:
"The retrieved logs do not contain enough information to answer this question."

Do NOT infer causality from timestamps or event order.
Do NOT claim that one event caused another unless the logs explicitly state this.

When asked about chronological order, use the timestamps from the retrieved logs.
"""
)

print("RAG-agent geladen.")

Goed. De agent is nu weer volledig opgebouwd.

Stap 5 — De agent testen

We testen nu of de volledige keten werkt:

Vraag → agent → search_logs → Chroma → relevante logs → antwoord

Waarom deze stap: hiermee controleren we dat de agent daadwerkelijk de bestaande vector database gebruikt en niet opnieuw de originele logfile hoeft te lezen of embeddings hoeft te maken.

Stap 5a — Retriever opnieuw aanmaken

Waarom deze stap: search_logs gebruikt retriever.invoke(...). Zonder retriever kan de tool niet bij de Chroma-database komen.

Stap 5a — Chroma opnieuw laden

Waarom deze stap: vectorstore is de Python-koppeling naar de bestaande database op ./log_parser_db. We openen de bestaande database opnieuw; er worden geen embeddings gemaakt.

In [ ]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

db_path = Path("./log_parser_db")

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest"
)

vectorstore = Chroma(
    collection_name="log_parser",
    embedding_function=embeddings,
    persist_directory="./log_parser_db"
)

print("Bestaande Chroma database geladen.")

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("Retriever geladen.")

Goed. Nu kunnen we eerst controleren of de retriever zelfstandig werkt.

Stap 5c — Retriever testen

Waarom deze stap: we testen eerst de databaseketen los van de agent. Als dit werkt, weten we dat vectorstore → retriever correct is.

In [ ]:
docs = retriever.invoke("What errors occurred?")

for doc in docs:
    print(doc.page_content)
    print("---")

oed. De retriever werkt weer correct. De bestaande Chroma-database wordt daadwerkelijk gebruikt.

Stap 5d — search_logs opnieuw testen

Waarom deze stap: we controleren nu de tussenlaag search_logs. Daarna kunnen we de agent opnieuw testen.

In [ ]:
result = search_logs.invoke({
    "question": "What errors occurred?"
})

print(result)

Goed. search_logs werkt nu ook. De hele retrieval-keten is daarmee gecontroleerd:

Chroma → retriever → search_logs

Stap 5e — Agent opnieuw testen

Je agent_rag bestaat nog steeds, maar nu zijn ook alle afhankelijkheden (vectorstore, retriever en search_logs) weer beschikbaar.

Waarom deze stap: nu testen we de volledige keten: de agent krijgt de vraag, roept search_logs aan, haalt de gegevens uit de bestaande Chroma-database en formuleert het antwoord.

In [ ]:
response = agent_rag.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What errors occurred in the logs?"
        }
    ]
})

print(response["messages"][-1].content)

Dit is correct. De agent heeft de drie ERROR-regels uit de logs geïdentificeerd en de timestamps behouden. Er zijn geen nieuwe gebeurtenissen verzonnen.

Daarmee is het belangrijkste doel van deze les bereikt: de agent gebruikt nu een persistente vector database en hoeft de logs niet telkens opnieuw te embedden.

We kunnen nog één laatste controle doen die direct aansluit op de les: de code opnieuw uitvoeren om te bevestigen dat de bestaande database wordt hergebruikt.

Stap 6 — Controleren dat embedding wordt overgeslagen

Waarom deze stap: dit demonstreert precies de optimalisatie uit de les: als log_parser_db al bestaat, worden de embeddings niet opnieuw gemaakt.

In [ ]:
if db_path.exists():
    print("Vector database bestaat al. Embedding wordt overgeslagen.")
else:
    print("Vector database bestaat nog niet. Embeddings moeten worden gemaakt.")

a, dit is les 57.

De kern van deze les is niet een nieuwe functionaliteit bouwen, maar de bestaande log-reader/RAG-code aanpassen aan de LangChain 1.0-API.

Voor jouw notebook betekent dat vooral:

OllamaEmbeddings gebruiken voor embeddings.
langchain_text_splitters gebruiken voor de text splitter.
De bestaande Chroma/RAG-logica behouden.
Voor de agent kan de cursus de _classic-variant gebruiken, terwijl wij al de LangChain 1.0-aanpak met create_agent() gebruiken.
Het model kan in de cursus veranderen naar een Ollama Cloud-model, maar dat is voor jouw lokale setup niet noodzakelijk.

Wij hebben een belangrijk deel van deze migratie al gedaan. Onze huidige code gebruikt bijvoorbeeld:

Wij hebben een belangrijk deel van deze migratie al gedaan. Onze huidige code gebruikt bijvoorbeeld:

from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.agents import create_agent

Dus voor les 57 hoeven we niet opnieuw alles op te bouwen.

Stap 1 — Controleer onze huidige LangChain-versies

Waarom deze stap: de les gaat specifiek over de overgang naar LangChain 1.0. We controleren daarom eerst of jouw omgeving nog steeds de versies gebruikt waarmee we deze code hebben opgebouwd.

In [ ]:
import langchain
import langchain_core

print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)

Daarmee kunnen we les 57 uitvoeren zonder terug te gaan naar de oudere syntax uit de video. Stap 2 — Controleer de imports die les 57 behandelt. Waarom deze stap: de docent noemt precies de migraties van embeddings en text splitters. We controleren dat jouw huidige code deze LangChain 1.x-imports al gebruikt.

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.agents import create_agent

print("LangChain 1.x imports werken.")

Goed. De imports uit les 57 werken. We hoeven dus geen oude _classic-syntax uit de video over te nemen. Stap 3 — Controleer de Ollama-embedding. Waarom deze stap: de docent noemt OllamaEmbeddings als één van de wijzigingen. We controleren of jouw embedding-model nog correct beschikbaar is.

In [ ]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest"
)

test_embedding = embeddings.embed_query("test")

print("Embedding werkt.")
print("Aantal dimensies:", len(test_embedding))

Goed. Het embedding-gedeelte van les 57 werkt. Stap 4 — Controleer de text splitter. Waarom deze stap: dit is de tweede concrete LangChain 1.x-wijziging uit de les. We controleren dat documenten nog steeds correct in chunks worden verdeeld.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

test_text = """
2026-09-21 10:15:32 INFO Application started
2026-09-21 10:15:35 INFO Connecting to MCP server
2026-09-21 10:15:40 ERROR MCP server connection timeout
"""

chunks = splitter.create_documents([test_text])

print("Aantal chunks:", len(chunks))
print(chunks[0].page_content)

Goed. De text splitter werkt correct en create_documents() levert een Document op, precies zoals we nodig hebben voor Chroma. Stap 5 — Controleer de LangChain 1.x agent. Waarom deze stap: de docent noemt de agentwijziging naar _classic, maar wij gebruiken al de nieuwe LangChain 1.x create_agent()-syntax. We controleren dat deze syntax in jouw omgeving werkt.

In [ ]:
from langchain.agents import create_agent

test_agent = create_agent(
    model=llm,
    tools=[search_logs]
)

print("LangChain 1.x agent werkt.")

Goed. Daarmee is les 57 praktisch afgerond. We hebben de drie wijzigingen uit de video gecontroleerd: OllamaEmbeddings werkt en geeft 768 dimensies. RecursiveCharacterTextSplitter werkt met create_documents(). create_agent() werkt met jouw LangChain 1.x-versie. Belangrijk: je hoeft de _classic-syntax uit de video niet over te nemen. De cursus gebruikt die om compatibiliteit/overgang eenvoudig te houden, terwijl jouw omgeving al met LangChain 1.x werkt. Ook de fout die de docent in de video noemt rond split_documents() hebben wij niet: wij gebruiken al correct create_documents(). Les 57: afgerond. We kunnen nu doorgaan met les 58 zodra je de transcriptie daarvan hebt.

Les 58 is de start van een nieuwe sectie: een BDD test case generator. De architectuur van deze les is iets anders dan onze log-analyzer. De basisgedachte is: Requirement PDF → Test Case Creator tool → Agent → BDD test cases. De gebruiker geeft bijvoorbeeld een requirementdocument en vraagt: Generate test cases in BDD format based on this requirement. De agent beslist vervolgens dat hij de test case creator-tool moet gebruiken. Die tool haalt de relevante informatie uit de requirementbestanden en geeft die terug aan het LLM. Het LLM genereert vervolgens de daadwerkelijke BDD-testgevallen. Voor jouw QA-werk is dit een relevante toepassing, omdat het concept direct aansluit op acceptance criteria en requirements. Voor deze eerste les hoeven we nog geen code te schrijven. De docent introduceert alleen het concept en de architectuur. De belangrijke onderdelen van les 58 zijn: Requirementbestand, bijvoorbeeld PDF. Een test_case_creator-tool. Tool calling door de agent. Requirementinformatie als input voor het LLM. BDD-output, bijvoorbeeld Given / When / Then. Uiteindelijk kan de output als testartefact worden opgeslagen. Eén belangrijk punt uit de video: de docent presenteert het als oplossing voor organisaties met veel verspreide requirementdocumenten. Het gaat dus niet alleen om "een PDF aan een LLM geven", maar vooral om een agent die via tools toegang krijgt tot relevante bedrijfsdocumentatie. We hoeven nu nog niets te bouwen. Stuur de volgende transcriptie zodra de docent daadwerkelijk met de implementatie begint.

De docent bouwt in deze les alleen het eerste deel: een generate_test_cases-tool met een herbruikbare prompt template. De LLM wordt in deze les nog niet aan de prompt gekoppeld; dat komt in les 60. We gaan dit in jouw LangChain 1.4.2-omgeving doen. We gebruiken dus de huidige LangChain-syntax en niet eventuele verouderde imports uit de video. Stap 1 — Maak het LLM klaar. De docent begint ook hiermee. Waarom deze stap: de test case generator heeft straks een LLM nodig om de user stories om te zetten naar BDD-testcases. Gebruik jouw bestaande lokale model:


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0
)

print("LLM geladen.")

Goed. De LLM is klaar. We gaan nu stap voor stap verder met les 59.

Stap 2 — Maak de generate_test_cases tool.

Waarom deze stap: de tool wordt straks door de agent gebruikt om een user story om te zetten naar een prompt voor het genereren van BDD-testcases. In deze les koppelen we de LLM nog niet aan de tool; dat komt in les 60.

In [13]:
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate

@tool
def generate_test_cases(user_story: str) -> str:
    """Generate BDD test cases from a user story."""

    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Your task is to convert the following user story into at least ten test cases.

Include combinations of:
- Valid scenarios
- Invalid scenarios
- Edge cases
- Alternative scenarios

Write the test cases in Gherkin BDD style.

User story:
{user_story}

Use the following BDD format:

Feature: [feature name]

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]
""")

    return prompt.invoke({
        "user_story": user_story
    }).to_string()

print("generate_test_cases tool aangemaakt.")

generate_test_cases tool aangemaakt.


Stap 3 — Test de tool met een eenvoudige user story.

Waarom deze stap: we controleren of de tool de user story correct in het prompt-template plaatst. We verwachten hier nog géén daadwerkelijke testcases; de LLM is nog niet aan de tool gekoppeld. Dat gebeurt in les 60.

In [ ]:
user_story = """
As a customer,
I want to log in with my email and password,
so that I can access my account.
"""

result = generate_test_cases.invoke({
    "user_story": user_story
})

print(result)

Dit is precies wat we op dit moment verwachten. De tool heeft de user story in het prompt-template geplaatst. Les 59 is hiermee praktisch afgerond: de tool en prompt-template werken. De daadwerkelijke generatie van de testcases door de LLM komt in les 60. Stuur de transcriptie van les 60, dan gaan we vanaf daar verder.

Dit is les 60. Hier wordt de vorige tool daadwerkelijk aan de LLM gekoppeld en daarna wordt de agent gebouwd die de tool kan gebruiken.

We doen het weer stap voor stap.

Stap 1 — Laat de generate_test_cases tool de LLM daadwerkelijk aanroepen.

Waarom deze stap: tot nu toe gaf onze tool alleen het prompt terug. Nu moet de prompt naar llm.invoke(...), zodat de LLM daadwerkelijk de BDD-testcases genereert.

Pas je bestaande tool aan naar:

In [ ]:
@tool
def generate_test_cases(user_story: str) -> str:
    """Generate BDD test cases from a user story."""

    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Your task is to convert the following user story into at least ten test cases.

Include combinations of:
- Valid scenarios
- Invalid scenarios
- Edge cases
- Alternative scenarios

Write the test cases in Gherkin BDD style.

User story:
{user_story}

Use the following BDD format:

Feature: [feature name]

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]
""")

    prompt_value = prompt.invoke({
        "user_story": user_story
    })

    response = llm.invoke(prompt_value)

    return response.content

geen foutmelding

Goed. Dan werkt de tool nu ook daadwerkelijk met de LLM.

Stap 2 — Test de generate_test_cases tool rechtstreeks.

Waarom deze stap: voordat we er een agent omheen bouwen, controleren we eerst of de tool zelfstandig daadwerkelijk BDD-testcases kan genereren. Dit maakt het later makkelijker om een probleem met de tool of met de agent te onderscheiden.

Gebruik dezelfde user story als in de les:

In [ ]:
user_story = """
As a user,
I want to reset my password using a link sent to my email,
so that I can regain access to my account if I forgot my password.
"""

result = generate_test_cases.invoke({
    "user_story": user_story
})

print(result)

👍 We gaan verder vanaf je bestaande generate_test_cases-tool. We bouwen niets opnieuw.
Stap 1 — Controleer eerst je bestaande tool
Waarom deze stap: We willen eerst zeker weten dat je huidige generate_test_cases-tool nog werkt voordat we hem aan de agent koppelen.
Voer je bestaande code uit:

Als je opnieuw de BDD-testcases terugkrijgt, zijn we klaar voor de volgende stap.
Stap 2 — De bestaande tool aan de agent geven
Waarom deze stap: Tot nu toe roep jij de tool zelf aan met .invoke(). Nu gaan we de agent de beslissing laten nemen om generate_test_cases te gebruiken.
De kern wordt:

Daarna maken we de agent, vergelijkbaar met de agents die we eerder hebben gemaakt.
Maar voordat we deze stap uitvoeren: stuur mij even je huidige code waarin je generate_test_cases hebt gedefinieerd. Dan sluiten we exact aan op jouw bestaande notebook en gebruiken we dezelfde LLM, prompt en imports.De lijst tools wordt pas gebruikt wanneer we hem meegeven aan een agent. We hoeven dus niet te verwachten dat deze regel zelf iets print.

In [ ]:
tools = [generate_test_cases]

In [ ]:
tools = [generate_test_cases]

print(tools)

Precies. Dit is goed. generate_test_cases staat nu als StructuredTool in je tools-lijst.
Waarom deze stap: De agent moet nu toegang krijgen tot deze tool. De volgende stap uit de les is daarom het maken van de agent. We gebruiken hiervoor jouw bestaande LLM en hoeven generate_test_cases niet opnieuw te definiëren.
Als je in de eerdere lessen create_tool_calling_agent hebt gebruikt, wordt het ongeveer

Precies. Dit is goed. generate_test_cases staat nu als StructuredTool in je tools-lijst.
Waarom deze stap: De agent moet nu toegang krijgen tot deze tool. De volgende stap uit de les is daarom het maken van de agent. We gebruiken hiervoor jouw bestaande LLM en hoeven generate_test_cases niet opnieuw te definiëren.
Als je in de eerdere lessen create_tool_calling_agent hebt gebruikt, wordt het ongeveer:

In [ ]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

Dit is geen fout in jouw generate_test_cases. Het probleem is dat de code uit de Udemy-les waarschijnlijk gebaseerd is op een oudere LangChain-versie. In jouw geïnstalleerde versie bestaat create_tool_calling_agent niet meer op die locatie.
We moeten dus niet zomaar verdergaan met code uit de video.
Waarom deze stap: We moeten eerst vaststellen welke LangChain-versie jij daadwerkelijk gebruikt en welke agent-API daarbij hoort. Anders blijven we oude cursuscode proberen die niet bij jouw installatie past.
Voer dit uit:

In [ ]:
import langchain

print(langchain.__version__)

Je gebruikt LangChain 1.4.2. Dat verklaart de fout: de cursus gebruikt een oudere agent-aanpak. In LangChain 1.x is create_agent de huidige eenvoudige manier om een agent met tools te bouwen.
We gaan dus dezelfde oefening doen, maar met de API die bij jouw versie hoort.
Waarom deze stap: create_agent vervangt hier de oudere create_tool_calling_agent. Je bestaande generate_test_cases blijft volledig hetzelfde.

In [ ]:
from langchain.agents import create_agent

In [ ]:
agent = create_agent(
    model=llm,
    tools=[generate_test_cases]
)

In [ ]:
print(agent)

Goed. Dat resultaat betekent dat de agent succesvol is aangemaakt. CompiledStateGraph is normaal bij create_agent() in LangChain 1.4.2. Je hebt nu dus: generate_test_cases ↓ tools ↓ create_agent() ↓ CompiledStateGraph.
Nu komt het belangrijke deel: de agent daadwerkelijk een opdracht geven.
Waarom deze stap: Tot nu toe hebben we alleen de agent geconfigureerd. Met invoke() geven we hem nu een user story en kan hij zelf bepalen dat hij de generate_test_cases-tool moet gebruiken.
Gebruik:

In [ ]:
user_story = """
As a user,
I want to reset my password using a link sent to my email,
so that I can regain access to my account if I forgot my password.
"""

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"Create BDD test cases for this user story:\n{user_story}"
        }
    ]
})

Daarna bekijken we de uiteindelijke output:

Waarom deze stap: LangChain 1.x geeft de agentuitvoer terug als een state met messages. We willen eerst zien wat de agent precies heeft gedaan voordat we de output verder formatteren.

In [ ]:
print(result["messages"][-1].content)

We gaan door met de volgende les: de agent moet nu een extern requirementsdocument kunnen lezen en op basis daarvan testcases genereren.
We houden jouw werkende agent en generate_test_cases intact.
Stap 1 — Eerst een document als input
Waarom deze stap: Tot nu toe staat de user story rechtstreeks in je Python-code. De volgende stap is dat de requirements uit een bestand komen.
Voor de oefening gebruiken we eerst een eenvoudig tekstbestand. Maak in je project bijvoorbeeld:
requirements.txt
met:

Stap 2 — Het bestand uitlezen

Waarom deze stap: Python moet eerst de inhoud van het requirementsbestand beschikbaar maken voordat we die aan de agent kunnen geven.

In [ ]:
with open("requirements.txt", "r", encoding="utf-8") as file:
    requirements = file.read()

print(requirements)

In [ ]:
import os

print(os.getcwd())

Dat verklaart de fout. Je notebook draait vanuit:

C:\Users\Mill\agentsforQA\.venv\Section6_PlaywrightToolkit

Daar staat waarschijnlijk geen requirements.txt. Je projectstructuur had eerder onder andere Section6_PlaywrightToolkit en een Docs-map. We moeten dus eerst kijken welke bestanden daar daadwerkelijk staan.

Waarom deze stap

We controleren de inhoud van de huidige map, zodat we het bestand kunnen gebruiken dat daadwerkelijk bij deze les hoort in plaats van zelf een bestand op de verkeerde plek aan te maken.

In [ ]:
import os

print(os.listdir())

Stap 1 — Eerst een document als input
Waarom deze stap: Tot nu toe staat de user story rechtstreeks in je Python-code. De volgende stap is dat de requirements uit een bestand komen.
Voor de oefening gebruiken we eerst een eenvoudig tekstbestand. Maak in je project bijvoorbeeld:
requirements.txt
met:
Password Reset Requirement
As a user, I want to reset my password using a link sent to my email,
so that I can regain access to my account if I forgot my password.
The password reset link must:
- Be sent to the user's registered email address.
- Expire after a defined period.
- Reject an invalid or expired token.
- Allow the user to enter a new password.
- Display an error when the email address is not registered.

Stap 2 — Het bestand uitlezen

Waarom deze stap: Python moet eerst de inhoud van het requirementsbestand beschikbaar maken voordat we die aan de agent kunnen geven.

In [ ]:
requirements_text = """Password Reset Requirement

As a user, I want to reset my password using a link sent to my email,
so that I can regain access to my account if I forgot my password.

The password reset link must:
- Be sent to the user's registered email address.
- Expire after a defined period.
- Reject an invalid or expired token.
- Allow the user to enter a new password.
- Display an error when the email address is not registered.
"""

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements_text)

print("requirements.txt created.")

In [ ]:
with open("requirements.txt", "r", encoding="utf-8") as file:
    requirements = file.read()

print(requirements)

Goed. Het bestand wordt nu correct ingelezen en de requirements zijn beschikbaar in de variabele requirements.
De volgende stap is om deze requirements aan je al bestaande agent te geven, zodat de agent op basis van het document de BDD-testcases genereert.
Waarom deze stap
Hiermee testen we precies het doel van deze oefening: de agent krijgt requirements uit een extern bestand als input en gebruikt vervolgens je bestaande generate_test_cases tool.

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"""
Read the following software requirements and generate BDD test cases.

Requirements:
{requirements}
"""
        }
    ]
})

Waarom deze stap

agent.invoke() geeft het volledige agentresultaat terug. De uiteindelijke tekst van de agent staat in het laatste bericht.

In [ ]:
print(result["messages"][-1].content)

Dit werkt. De agent heeft de requirements uit requirements.txt gelezen en op basis daarvan BDD-testcases gegenereerd.
De melding Output is truncated is alleen een Jupyter-weergaveprobleem. Het betekent niet dat de agent-output ontbreekt.
Als je de volledige output wilt bekijken, kun je dezelfde output in een scrollbaar tekstveld laten tonen:
Waarom deze stap
Zo controleren we de volledige gegenereerde testcases zonder de agent opnieuw uit te voeren.

In [ ]:
from IPython.display import display, Markdown

display(Markdown(result["messages"][-1].content))

Ja. Dit is een geldige output van de agent en de oefening is hiermee geslaagd.
Je ziet ook iets interessants in de output: de agent heeft niet alleen de expliciet genoemde requirements vertaald, maar zelf extra scenario's toegevoegd rond case sensitivity, special characters en URL encoding. Dat laat zien dat de LLM de requirements interpreteert en uitbreidt, in plaats van alleen letterlijk de regels te kopiëren.
Voor de cursus is vooral belangrijk dat deze keten nu werkt:
requirements.txt → requirements inlezen → agent.invoke() → generate_test_cases → BDD-testcases
Je hoeft hier dus niets meer aan te veranderen.
De volgende stap is waarschijnlijk de volgende cursusles, waarin de agent een extern document gaat lezen. Als je de transcriptie daarvan stuurt, volgen we die vanaf daar.

We gaan het nu praktisch doen. We passen jouw bestaande agentsforQA-project stap voor stap aan, maar ik wil eerst de huidige situatie controleren voordat we code gaan wijzigen.
Stap 1 — Open je project
Open in VS Code:
C:\Users\Mill\agentsforQA
Open vervolgens je notebook of Python-bestand waarin je huidige QA-agent staat.
Waarom deze stap
We gaan de bestaande code aanpassen. We willen voorkomen dat we een tweede, afwijkende implementatie maken.

Stap 2 — Controleer je Python-omgeving

Open de terminal in VS Code en activeer je virtual environment:

Stap 5 — Gemini-pakketten installeren

Waarom deze stap

Voor de les hebben we twee pakketten nodig: langchain-google-genai om Gemini vanuit LangChain aan te spreken en python-dotenv om de API-key veilig vanuit een .env-bestand te laden.

Voer uit:

In [ ]:
%pip install -U langchain-google-genai python-dotenv

Stap 6 — API-key in .env
Waarom deze stap
De Gemini API-key moet niet rechtstreeks in je Python-code staan. We zetten hem in een .env-bestand, zodat je hem niet per ongeluk naar GitHub pusht.
Je hebt hiervoor een Google AI Studio API-key nodig. Als je die al hebt, maak dan in:
C:\Users\Mill\agentsforQA
een bestand aan met exact deze naam:
.env
Zet daarin:
GOOGLE_API_KEY=jouw_api_key_hier
Vervang jouw_api_key_hier door je echte key.
Belangrijk: zet de API-key niet hier in de chat.
Daarna moeten we controleren of .env ook in .gitignore staat.
Waarom deze stap
Je project staat op GitHub. De .env mag daarom nooit als onderdeel van je Git-repository worden geüpload.
Voer in PowerShell uit:

Goed. .env staat al correct in .gitignore. De API-key kan dus lokaal blijven en wordt niet door Git meegenomen.
Stap 7 — controleren of je .env correct wordt geladen
Waarom deze stap
Voordat we Gemini aanroepen, testen we alleen of Python de API-key uit .env kan lezen. We printen bewust niet de key zelf.
Voer dit uit in een Jupyter Notebook-cell:

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

print("API key geladen:", bool(api_key))

Dan wordt .env niet gevonden, of de variabele heet anders. We controleren eerst waar je notebook en .env zich bevinden. We gaan de API-key zelf niet tonen.
Stap 7a — controleren vanuit welke map Jupyter draait
Waarom deze stap
load_dotenv() zoekt standaard naar een .env-bestand vanuit de huidige werkomgeving. We moeten daarom eerst weten welke map Python momenteel gebruikt.
Voer in je Jupyter Notebook uit:

In [ ]:
import os

print("Huidige map:")
print(os.getcwd())

print("\n.env aanwezig:")
print(os.path.exists(".env"))

Daar zit het probleem. Je Jupyter Notebook draait vanuit:
C:\Users\Mill\agentsforQA\.venv\Section6_PlaywrightToolkit
maar je .env hoort in de projectroot te staan:
C:\Users\Mill\agentsforQA\.env
Je gebruikt dus een andere werkmap dan de map waarin .env staat.
Stap 7b — controleren of .env in de projectroot staat
Waarom deze stap
We willen eerst bevestigen dat het .env-bestand daadwerkelijk op de juiste plek staat. We lezen de API-key niet uit.
Voer in Jupyter uit:

In [ ]:
from pathlib import Path

env_file = Path(r"C:\Users\Mill\agentsforQA\.env")

print("Bestaat .env:", env_file.exists())

Dan is de oorzaak duidelijk: het .env-bestand bestaat nog niet in de projectroot. We maken het nu aan.
Stap 7c — .env aanmaken
Waarom deze stap
De Python-code kan alleen een API-key laden als er daadwerkelijk een .env-bestand bestaat. We plaatsen het bestand direct in C:\Users\Mill\agentsforQA, dus niet in .venv of Section6_PlaywrightToolkit.
Open PowerShell en controleer eerst dat je hier staat:

Ja. Als je specifiek langchain-google-genai==2.1.6 wilt gebruiken, kunnen we daarmee werken. De syntax voor Gemini 2.5 Flash blijft:

Waarom deze stap: hiermee zorgen we dat je exact dezelfde LangChain Google GenAI-integratie gebruikt als in jouw omgeving, terwijl we het model op Gemini 2.5 Flash zetten.Daarna:

Waarom deze stap: hiermee controleren we of de package correct is geïnstalleerd en gebruiken we Gemini 2.5 Flash via LangChain.

In [ ]:
from dotenv import dotenv_values

config = dotenv_values(r"C:\Users\Mill\agentsforQA\.env")

print(list(config.keys()))

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(r"C:\Users\Mill\agentsforQA\.env", override=True)

print("API key gevonden:", bool(os.getenv("GOOGLE_API_KEY")))

In [ ]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(r"C:\Users\Mill\agentsforQA\.env", override=True)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

result = llm.invoke("Say hello in one sentence.")
print(result.content)

Dat bevestigt dat de Gemini API-koppeling werkt.

Je hebt nu:

.env → GOOGLE_API_KEY correct geladen
langchain-google-genai==2.1.6 → werkt
Gemini → bereikbaar
llm.invoke() → werkt
result.content → geeft het modelantwoord terug

We kunnen nu verder met de cursus en de generate_test_cases-tool bouwen.

Dan gaan we verder met de les waarin de LLM via een tool BDD-testcases genereert.

Stap 1 — De LLM als functie testen

Waarom deze stap

In de cursus wordt eerst een aparte functie gemaakt die een user story ontvangt en Gemini vraagt om daar BDD/Gherkin-testcases van te maken. We testen dit eerst los van de agent, zodat we weten dat de LLM-functionaliteit werkt voordat we er een tool omheen bouwen.

In [ ]:
from dotenv import load_dotenv
import os

from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(r"C:\Users\Mill\agentsforQA\.env", override=True)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

def generate_test_cases(user_story: str):
    prompt = f"""
You are a senior software tester.

Generate BDD test cases in Gherkin format for the following user story.

User story:
{user_story}

Create at least 10 different test scenarios.
Cover positive, negative, boundary and validation cases.

Use this structure:

Feature:
Scenario:
  Given
  When
  Then
"""

    response = llm.invoke(prompt)
    return response.content

Stap 2 — Een user story meegeven

Waarom deze stap

Hiermee controleren we of Gemini daadwerkelijk bruikbare BDD-scenario's genereert. Dit is dezelfde basisfunctionaliteit die later door de agent via een tool wordt aangeroepen.

In [ ]:
user_story = """
As a user,
I want to reset my password using my email address,
so that I can regain access to my account if I forget my password.
"""

test_cases = generate_test_cases(user_story)

print(test_cases)

Dit is een goede uitkomst van de eerste test. Gemini genereert daadwerkelijk bruikbare Gherkin-scenario's en volgt de instructie om positieve, negatieve, boundary- en securitygevallen mee te nemen.

Eén belangrijk punt: sommige details zijn door Gemini zelf ingevuld, bijvoorbeeld een maximale wachtwoordlengte van 64 tekens en een resetlink die na 24 uur verloopt. Die waarden zijn alleen correct als ze daadwerkelijk in de requirements staan. Voor echte QA moeten ze dus uit de requirements komen en niet door de LLM worden verzonnen.

Voor de cursus kunnen we nu naar de volgende laag: de functie beschikbaar maken als tool voor de agent.

Stap 3 — generate_test_cases als LangChain-tool

Waarom deze stap

Tot nu toe roepen wij generate_test_cases() zelf aan. De bedoeling van de agent-architectuur is dat de agent deze functie als tool kan gebruiken wanneer de gebruiker daarom vraagt.

In [ ]:
from langchain_core.tools import tool

@tool
def generate_test_cases(user_story: str) -> str:
    """
    Generate BDD/Gherkin test cases for a given user story.
    """

    prompt = f"""
You are a senior software tester.

Generate BDD test cases in Gherkin format for the following user story.

User story:
{user_story}

Create at least 10 different test scenarios.
Cover positive, negative, boundary and validation cases.

Use this structure:

Feature:
Scenario:
  Given
  When
  Then
"""

    response = llm.invoke(prompt)
    return response.content

Stap 4 — Controleren of de tool bestaat

Waarom deze stap

Voordat we een agent bouwen, controleren we eerst of LangChain de functie daadwerkelijk als tool heeft geregistreerd.

In [ ]:
print(generate_test_cases.name)
print(generate_test_cases.description)

Precies. De tool is correct geregistreerd.

Nu gaan we de volgende stap doen: de agent toegang geven tot deze tool.

Stap 5 — De tool aan de agent koppelen

Waarom deze stap

De agent moet weten welke acties hij mag uitvoeren. We geven hem daarom generate_test_cases als beschikbare tool. De agent kan daarna bepalen wanneer deze tool nodig is.

In [ ]:
tools = [generate_test_cases]

llm_with_tools = llm.bind_tools(tools)

print(llm_with_tools)

Dit is correct. De tool is nu daadwerkelijk aan Gemini gekoppeld.

De belangrijke regel in je output is:

kwargs={'tools': [{'type': 'function', 'function': {'name': 'generate_test_cases', ...}}]}

Dat betekent dat Gemini de tool generate_test_cases kent en weet dat deze één verplichte parameter heeft:

user_story

We zijn nu klaar met het configureren van de tool. Nu moeten we testen of Gemini hem ook daadwerkelijk aanroept.

Stap 6 — Gemini de tool laten aanroepen

Waarom deze stap

Bij de vorige stap hebben we alleen gecontroleerd of de tool beschikbaar is. Nu geven we Gemini een opdracht waarbij het model zelf moet bepalen dat generate_test_cases gebruikt moet worden.

Voer dit uit:

In [ ]:
request = """
Create BDD test cases for this user story:

As a registered user,
I want to reset my password using my email address,
so that I can regain access to my account if I forget my password.
"""

response = llm_with_tools.invoke(request)

print(response)

Dit is precies de output die we wilden zien. Gemini heeft de tool geselecteerd en de juiste user_story als argument meegegeven.

De belangrijkste regel is:

tool_calls=[{
    'name': 'generate_test_cases',
    'args': {
        'user_story': 'As a registered user, ...'
    }
}]

Er is alleen nog één stap nodig: de tool daadwerkelijk uitvoeren. Tot nu toe heeft Gemini gezegd: "gebruik deze tool", maar Python heeft de tool nog niet uitgevoerd.

Stap 7 — De tool-call daadwerkelijk uitvoeren

Waarom deze stap

We halen eerst de tool-call uit de response. Daarna geven we de meegeleverde user_story door aan onze Python-functie generate_test_cases(). Nu zou je opnieuw de Gherkin-testcases moeten krijgen.

De architectuur is dan:

User
  ↓
Gemini
  ↓
Gemini kiest generate_test_cases
  ↓
tool_call
  ↓
Python voert generate_test_cases uit
  ↓
Gemini genereert BDD/Gherkin testcases

Dit is echter nog niet de volledige agent-loop. Een echte agent geeft het resultaat van de tool namelijk weer terug aan het model, zodat het model vervolgens een definitief antwoord aan de gebruiker kan formuleren.

Dat is de volgende stap in de cursus.

In [ ]:
tool_call = response.tool_calls[0]

user_story = tool_call["args"]["user_story"]

tool_result = generate_test_cases.invoke({
    "user_story": user_story
})

print(tool_result)

Dit bevestigt dat de tool correct is uitgevoerd. Je hebt nu de volledige keten werkend tot en met het genereren van de testcases.

Er is wel een belangrijk QA-punt zichtbaar in deze output: Gemini verzint opnieuw requirements die niet in de user story stonden, zoals "24 hours", "8 characters", rate limiting van 3 verzoeken in 5 minuten en 15 minuten blokkeren. Voor deze cursus is dat prima als demonstratie van de tool, maar voor jouw uiteindelijke QA-agent willen we dit juist voorkomen.

De volgende stap is de echte agent-loop.

Stap 8 — Toolresultaat teruggeven aan Gemini

Waarom deze stap

Nu hebben we:

Gemini → tool kiezen → Python voert tool uit → testcases

Maar Gemini heeft het resultaat van de tool nog niet teruggekregen. We sturen daarom het resultaat terug naar het model. Daardoor kan Gemini uiteindelijk zelf een antwoord aan de gebruiker geven.

Gebruik:

In [ ]:
from langchain_core.messages import ToolMessage

tool_message = ToolMessage(
    content=tool_result,
    tool_call_id=tool_call["id"]
)

final_response = llm_with_tools.invoke([
    {
        "role": "user",
        "content": request
    },
    response,
    tool_message
])

print(final_response.content)

De fout is duidelijk. Dit ligt niet aan je API-key of aan generate_test_cases. Het probleem ontstaat doordat we de Gemini-response handmatig opnieuw opbouwen.

Gemini 3 vereist bij function calling dat de thought_signature van de oorspronkelijke function call behouden blijft wanneer je de conversatie naar Gemini terugstuurt. Google documenteert expliciet dat deze signature bij Gemini 3 verplicht is.

Wij deden dit:

In [ ]:
[
    {"role": "user", "content": request},
    response,
    tool_message
]

Stap 8 — Controleer de tool-call metadata

Waarom deze stap

We willen controleren of de originele AIMessage inderdaad de interne Gemini-metadata bevat die nodig is om de tool-call opnieuw aan Gemini aan te bieden.

Voer uit:

In [ ]:
print(response.additional_kwargs)

Dat verklaart de fout.

Je response.additional_kwargs bevat alleen:

{
    'function_call': {
        'name': 'generate_test_cases',
        'arguments': '...'
    }
}

Er staat geen thought_signature in. Daardoor kan Gemini 3 de oorspronkelijke tool-call niet opnieuw accepteren wanneer we het toolresultaat terugsturen.

Dit is dus geen fout in jouw generate_test_cases-functie. Het zit in de combinatie van Gemini 3 tool calling en de manier waarop jouw huidige LangChain-versie de response teruggeeft.

We moeten daarom niet verder proberen deze handmatige loop te repareren.

Stap 8 — Eerst controleren welke LangChain-versie je precies gebruikt

Waarom deze stap

Je gebruikt bewust langchain-google-genai==2.1.6. We moeten controleren of deze versie ondersteuning heeft voor de Gemini 3 thought-signature flow voordat we de agent verder bouwen.

Voer uit:

In [ ]:
import langchain_google_genai

print(langchain_google_genai.__version__)

Dat is geen probleem met je installatie. langchain_google_genai heeft in deze versie geen __version__-attribuut.

We gebruiken de betrouwbare Python package-metadata.

Stap 8 — Versie controleren

Waarom deze stap

Hiermee lezen we de daadwerkelijk geïnstalleerde versie uit de Python-omgeving waarin je notebook draait.


In [ ]:
from importlib.metadata import version

print(version("langchain-google-genai"))

Dan is dat bevestigd: je gebruikt inderdaad langchain-google-genai 2.1.6.

De kern is nu dat we met jouw huidige combinatie handmatig geen tweede Gemini-aanroep moeten doen na de tool-call. De thought_signature die Gemini 3 daarvoor nodig heeft, wordt in jouw huidige response niet beschikbaar gemaakt.

We kunnen dit eenvoudiger oplossen door de agent-loop door LangChain te laten beheren.

Stap 9 — Eerst de benodigde agent-functionaliteit controleren

Waarom deze stap

We willen voorkomen dat we weer een codevoorbeeld gebruiken dat niet past bij jouw geïnstalleerde LangChain-versies. Daarom controleren we eerst welke agentfunctie jouw omgeving daadwerkelijk beschikbaar heeft.


In [ ]:
import langchain

print(langchain.__version__)

Stap 1 — Installeer LangChain

Waarom deze stap

We hebben langchain nodig voor de agentlaag die de tool-call automatisch kan afhandelen. Je huidige langchain-google-genai is alleen de Gemini-integratie; daarom werkt from langchain_google_genai import ChatGoogleGenerativeAI wel, maar import langchain niet.

Voer in je actieve .venv312 uit:

In [ ]:
pip install langchain

Hier zit nu een belangrijke dependency-conflict in. langchain 1.4.2 heeft langchain-core 1.6.5 geïnstalleerd, maar jouw werkende langchain-google-genai 2.1.6 vereist juist langchain-core <0.4.0.

Dus: niet verdergaan met deze installatie. Anders breken we de Gemini-integratie die net werkte.

We moeten eerst een compatibele combinatie kiezen.

Stap 1 — Controleer wat er nu geïnstalleerd is

Waarom deze stap

We willen exact vaststellen welke versies pip nu heeft staan voordat we iets terugzetten.

In [ ]:
import importlib.metadata as md

for package in [
    "langchain",
    "langchain-core",
    "langchain-google-genai",
    "langgraph",
]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NIET GEÏNSTALLEERD")

Dat bevestigt het probleem. Je hebt nu twee incompatibele dependency-lijnen:

langchain-google-genai 2.1.6 → vereist langchain-core >=0.3.66,<0.4.0
langchain 1.4.2 → heeft langchain-core 1.6.5 geïnstalleerd

Omdat jouw Gemini-integratie met langchain-google-genai 2.1.6 al werkte, zou ik die behouden en langchain terugzetten naar de oudere 0.3-lijn. Dat past ook beter bij de code uit je cursus.

Stap 1 — Zet LangChain terug naar een compatibele versie

Waarom deze stap

We brengen langchain en langchain-core terug naar dezelfde generatie als jouw werkende langchain-google-genai 2.1.6. Ik zou hier expliciet pinnen, zodat pip niet opnieuw automatisch een incompatibele langchain-core installeert.

Voer dit uit in je actieve .venv312:

Als pip tijdens deze installatie waarschuwingen geeft over langgraph, is dat op dit moment niet het belangrijkste. We gebruiken langgraph nog niet voor deze stap.

Stap 2 — Controleer de versies

Waarom deze stap

We controleren eerst of de dependency-combinatie nu klopt voordat we opnieuw met de agent gaan werken.

In [ ]:
import importlib.metadata as md

for package in [
    "langchain",
    "langchain-core",
    "langchain-google-genai",
]:
    print(f"{package}: {md.version(package)}")

Nu kunnen we terug naar het oorspronkelijke probleem: de handmatige ToolMessage-route gaf bij Gemini een thought_signature-fout. We gaan die route niet handmatig voortzetten. LangChain kan de tool-call en het tool-resultaat voor ons afhandelen.

Stap 1 — Controleer eerst de agent-import

Waarom deze stap

LangChain 0.3.x gebruikt de klassieke AgentExecutor-architectuur. We controleren eerst of de benodigde agentfuncties in jouw installatie beschikbaar zijn voordat we je bestaande code aanpassen.

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent

print("Agent imports werken")

Stap 2 — Maak de prompt voor de agent

Waarom deze stap

De agent heeft een aparte prompt nodig waarin staat hoe hij met de tool moet werken. agent_scratchpad is belangrijk: LangChain gebruikt dit om de tussenstappen van de agent bij te houden.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a software testing assistant.
Use the available tool to generate BDD/Gherkin test cases
when the user asks for test cases."""
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

Stap 3 — Maak de agent

Waarom deze stap

Hier koppelen we drie onderdelen aan elkaar: jouw Gemini-LLM, de generate_test_cases-tool en de prompt. De agent kan hierdoor zelf bepalen wanneer hij de tool moet aanroepen.

In [ ]:
agent = create_tool_calling_agent(
    llm,
    [generate_test_cases],
    prompt
)

print("Agent aangemaakt.")

Stap 4 — Maak de AgentExecutor

Waarom deze stap

De agent bepaalt wat er moet gebeuren, maar de AgentExecutor voert de volledige cyclus uit: verzoek → tool kiezen → tool uitvoeren → resultaat teruggeven aan de agent → eindantwoord.

In [ ]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=[generate_test_cases],
    verbose=True
)

print("AgentExecutor aangemaakt.")

Stap 5 — Laat de agent de tool zelfstandig uitvoeren

Waarom deze stap

Tot nu toe hebben we alleen de onderdelen opgebouwd. Met deze stap geven we de agent een opdracht. De agent moet zelf herkennen dat generate_test_cases nodig is, de tool aanroepen en daarna het resultaat teruggeven.

Gebruik dezelfde user story als eerder:

In [ ]:
request = """
Create BDD test cases for this user story:

As a registered user,
I want to reset my password using my email address,
so that I can regain access to my account if I forget my password.
"""

result = agent_executor.invoke({
    "input": request
})

print(result["output"])

Dit werkt nu zoals bedoeld. De volledige agent-flow is succesvol uitgevoerd:
User request → Agent → Gemini → generate_test_cases → tool-resultaat → Agent → eindantwoord
Het belangrijke verschil met je eerdere poging is dat je de tool-resultaatcyclus nu niet meer zelf met ToolMessage hoeft op te bouwen. AgentExecutor regelt dat voor je. Daarmee is ook de eerdere thought_signature-fout verdwenen.
Er is wel één inhoudelijk QA-punt in de gegenereerde testgevallen. De LLM heeft eisen verzonnen die niet in jouw user story stonden, bijvoorbeeld: wachtwoord minimaal 8 tekens; verplichte hoofdletter, cijfer en speciaal teken; resetlink verloopt na 24 uur; maximaal 5 aanvragen per 10 minuten; blokkade van 15 minuten; specifieke foutmeldingen.
Dat is voor een echte QA-agent een belangrijk probleem: een LLM mag ontbrekende requirements niet als feit invullen. Die waarden moeten uit requirements, documentatie of andere betrouwbare bronnen komen.
En dat sluit precies aan op de volgende stap uit je cursus: requirements uit externe documentatie lezen in plaats van de user story als enige bron gebruiken.
Je huidige architectuur is nu:
User story ↓ Agent ↓ Gemini ↓ generate_test_cases tool ↓ LLM genereert testcases ↓ Agent geeft resultaat terug
De volgende stap wordt dan:
User story ↓ Agent ↓ Requirements/documentatie ophalen ↓ Requirements + user story ↓ generate_test_cases ↓ BDD testcases
Dat maakt je agent aanzienlijk betrouwbaarder voor QA, omdat de testgevallen dan gebaseerd kunnen worden op daadwerkelijk vastgelegde acceptatiecriteria in plaats van op aannames van Gemini.

Nu gaan we naar de volgende stap uit de cursus: de requirements uit externe documentatie laten ophalen voordat de testcases worden gegenereerd.

Je huidige agent werkt. We gaan daarom niet opnieuw beginnen.


Stap 1 — Maak eerst een requirements-tool

Waarom deze stap

Op dit moment krijgt generate_test_cases alleen de user story. Daardoor verzint Gemini ontbrekende regels. We maken eerst een tool die de requirements kan ophalen. In de volgende stap geven we die informatie door aan de test-case generator.

Voorlopig testen we dit met een eenvoudige tekstbron, zodat we eerst de agentarchitectuur goed krijgen.

In [ ]:
from langchain_core.tools import tool

@tool
def get_requirements() -> str:
    """
    Retrieve the functional requirements for the password reset feature.
    """

    return """
Password Reset Requirements:

1. Only registered users can request a password reset.
2. The user must provide a valid email address.
3. A password reset link is sent to the registered email address.
4. The reset link can only be used once.
5. The user must enter the new password twice.
6. The two passwords must match.
7. The new password must comply with the application's password policy.
"""

Stap 2 — Test de nieuwe tool

Waarom deze stap

We controleren eerst of de requirements-tool zelfstandig werkt voordat we hem aan de agent toevoegen.

In [ ]:
requirements = get_requirements.invoke({})

print(requirements)

Stap 3 — Voeg beide tools toe aan de agent

Waarom deze stap

De agent moet nu kunnen kiezen tussen:

get_requirements → requirements ophalen
generate_test_cases → BDD-testcases genereren

Daarna kan de agent zelf de volgorde bepalen.

In [ ]:
tools = [
    get_requirements,
    generate_test_cases
]

agent = create_tool_calling_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

print("Agent bijgewerkt met beide tools.")

Stap 4 — Test de agent

Waarom deze stap

We willen nu controleren of de agent daadwerkelijk beide tools gebruikt. De verbose=True instelling laat zien welke tool de agent kiest en in welke volgorde.

In [ ]:
request = """
Create BDD test cases for this user story:

As a registered user,
I want to reset my password using my email address,
so that I can regain access to my account if I forget my password.
"""

result = agent_executor.invoke({
    "input": request
})

print(result["output"])

Hier zien we precies wat er gebeurt. De eerste tool-call werkt: Invoking: generate_test_cases maar daarna probeert AgentExecutor de volgende Gemini-aanroep te doen en Gemini weigert de eerdere function call omdat de thought_signature ontbreekt. Dus het probleem zit niet in jouw get_requirements-tool. De agent gebruikt die zelfs nog niet. Gemini kiest direct generate_test_cases. Er zijn eigenlijk twee aparte punten die we moeten oplossen: Gemini moet de tool-call correct kunnen hergebruiken. De agent moet eerst get_requirements gebruiken voordat hij generate_test_cases aanroept. Gezien jouw huidige combinatie: langchain 0.3.27, langchain-core 0.3.86, langchain-google-genai 2.1.6, Gemini gemini-3.8-flash is de thought_signature-fout een compatibiliteitsprobleem tussen deze oudere LangChain/LangChain-Gemini-integratie en de huidige Gemini tool-calling API. Ik zou daarom niet verder sleutelen aan AgentExecutor en ook niet proberen de signature zelf te fabriceren. De beste volgende stap is eerst vaststellen welke versie van langchain-google-genai momenteel beschikbaar is en welke versie ondersteuning heeft voor de huidige Gemini tool-calling. Omdat dit actuele package/API-compatibiliteit betreft, wil ik dat eerst controleren voordat we packages gaan wijzigen. Waarom deze stap: we willen voorkomen dat we opnieuw een willekeurige combinatie installeren die elkaar niet ondersteunt. Je directe Gemini-call werkte al; die willen we behouden. Voer dit uit:

Dit bevestigt dat 2.1.6 behoorlijk achterloopt. Er is inmiddels 4.4.0, en belangrijker: Google documenteert dat Gemini 3 bij function calling thought_signature teruggeeft en dat die bij vervolgstappen moet worden meegestuurd.
Ook LangChain heeft in recente versies expliciet ondersteuning voor het bewaren/doorgeven van deze signatures.
We gaan daarom niet verder sleutelen aan je AgentExecutor. Eerst controleren we welke dependencies 4.4.0 nodig heeft.
Waarom deze stap
We moeten voorkomen dat we alleen langchain-google-genai upgraden en daardoor opnieuw een versieconflict krijgen met jouw huidige langchain 0.3.27 en langchain-core 0.3.86.
Voer dit uit in PowerShell:

Dit installeert nog niets. Het laat alleen zien welke dependencies pip zou willen aanpassen.

Stuur mij daarna de volledige output. Dan bepalen we de exacte combinatie van langchain, langchain-core en langchain-google-genai voordat we iets veranderen.

Dit ziet er nu correct uit. De omgeving is succesvol naar de nieuwe LangChain-stack gebracht:
langchain 1.4.2, langchain-core 1.6.5, langchain-google-genai 4.4.0, google-genai 2.25.0.
We moeten nu wel rekening houden met één belangrijke wijziging: jouw oude code met create_tool_calling_agent en AgentExecutor hoort bij de oude LangChain API. In LangChain 1.x is de aanbevolen agent-aanpak gewijzigd.
Waarom deze stap
Eerst controleren we of je Gemini-model en API-key nog correct werken met de nieuwe langchain-google-genai 4.4.0. We veranderen dus nog niets aan je agent.
Run in je notebook:

In [ ]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(r"C:\Users\Mill\agentsforQA\.env", override=True)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

response = llm.invoke("Say hello in one sentence.")

print(response.content)

De nieuwe stack werkt dus correct met Gemini. De volgende stap is je agent aanpassen aan LangChain 1.x. We gaan AgentExecutor en create_tool_calling_agent niet meer gebruiken.
Waarom deze stap
LangChain 1.x gebruikt de nieuwe create_agent() API. Daarmee wordt de tool-call geschiedenis door LangChain zelf beheerd, inclusief de informatie die Gemini nodig heeft voor vervolgstappen. Dat is precies het deel dat bij jouw oude setup de thought_signature-fout veroorzaakte.
Gebruik eerst alleen deze test met je bestaande get_requirements tool:

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[get_requirements]
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Retrieve the requirements for the password reset feature."
        }
    ]
})

print(result["messages"][-1].content)

Dat is nuttige informatie. De installatie zegt langchain 1.4.2, maar jouw geïnstalleerde langchain.agents bevat blijkbaar geen create_agent. We moeten nu niet opnieuw willekeurig packages wijzigen.

Waarom deze stap

We controleren eerst welke agent-functies jouw installatie daadwerkelijk beschikbaar stelt. Daarmee bepalen we de juiste API voor precies jouw geïnstalleerde versie.

Voer dit uit:

In [ ]:
import langchain
import langchain.agents as agents

print("LangChain:", langchain.__version__)
print("Agent functies:")
print([name for name in dir(agents) if "agent" in name.lower()])

In [ ]:
import langchain

print(langchain.__version__)

In [ ]:
import sys
import langchain

print("Python:")
print(sys.executable)

print("\nLangChain:")
print(langchain.__version__)

print("\nLangChain locatie:")
print(langchain.__file__)

In [ ]:
import sys
import importlib.metadata as metadata

print("Python:", sys.executable)
print("Geïnstalleerde LangChain:", metadata.version("langchain"))
print("Geïnstalleerde LangChain Core:", metadata.version("langchain-core"))
print("Geïnstalleerde Google GenAI:", metadata.version("langchain-google-genai"))

Dit bevestigt het probleem: de installatie is correct, maar de notebook heeft de oude langchain-module nog in het geheugen. importlib.metadata ziet 1.4.2, terwijl import langchain nog 0.3.27 teruggeeft.
We hoeven niets opnieuw te installeren.
Waarom deze stap
We moeten de Jupyter-kernel echt volledig beëindigen en opnieuw starten. Een gewone Python-import kan een al geladen module niet vervangen.
In VS Code:
Open de notebook.
Klik rechtsboven op de huidige kernelnaam.
Kies Shut Down Kernel als die optie beschikbaar is.
Sluit daarna de notebooktab.
Open de notebook opnieuw.
Selecteer expliciet:
c:\Users\Mill\agentsforQA\.venv312\Scripts\python.exe
Start de kernel opnieuw.
Voer daarna als eerste Python-cel alleen dit uit:

In [ ]:
import langchain
import sys

print("Python:", sys.executable)
print("LangChain:", langchain.__version__)
print("LangChain locatie:", langchain.__file__)

Dan hebben we nu een concreet installatieprobleem vastgesteld: de package-metadata zegt 1.4.2, maar de daadwerkelijke langchain-bestanden in site-packages zijn nog van 0.3.27. Dat hoort niet zo te zijn.
We gaan eerst de fysieke packagebestanden controleren voordat we opnieuw installeren.
Waarom deze stap
importlib.metadata leest de geïnstalleerde distributie-informatie uit de .dist-info map, terwijl Python bij import langchain de daadwerkelijke code uit de langchain-map laadt. Bij jou spreken die twee elkaar tegen.
Voer dit uit in PowerShell, dus niet in de notebook:


Dit is nu helder. De installatie zelf is goed. PowerShell ziet 1.4.2; alleen de huidige Jupyter-kernel blijft een oude langchain-module uit geheugen gebruiken.
We gaan daarom de kernel niet verder proberen te repareren. We kunnen de notebookkernel vanuit PowerShell geforceerd opnieuw starten door VS Code volledig te sluiten.
Waarom deze stap
De kernel is een afzonderlijk Python-proces. Zolang dat proces de oude langchain-module geladen heeft, kan de notebook 0.3.27 blijven tonen, ook al staat op disk inmiddels 1.4.2.
Doe nu het volgende:
Sla je notebook op.
Sluit alle VS Code-vensters.
Controleer eventueel in Taakbeheer dat er geen python.exe/jupyter-proces van deze notebook meer draait.
Start VS Code opnieuw.
Open:
C:\Users\Mill\agentsforQA
Open je notebook.
Selecteer opnieuw de kernel:
C:\Users\Mill\agentsforQA\.venv312\Scripts\python.exe

In [ ]:

import langchain
import sys

print("Python:", sys.executable)
print("LangChain:", langchain.__version__)

In [ ]:
import sys
import langchain

print("LangChain:", langchain.__version__)
print("LangChain locatie:", langchain.__file__)

print("\nPython search paths:")
for path in sys.path:
    print(path)

In [ ]:
from pathlib import Path
import langchain

version_file = Path(langchain.__file__).parent / "__init__.py"

print("Bestand:")
print(version_file)

print("\nInhoud van __init__.py:")
print(version_file.read_text(encoding="utf-8")[:3000])

In [ ]:
import importlib
import langchain

importlib.reload(langchain)

print("LangChain:", langchain.__version__)

from langchain.agents import create_agent

print("create_agent is beschikbaar")

In [ ]:
from pathlib import Path
import langchain.agents

agents_path = Path(langchain.agents.__file__).parent

print("Agents locatie:")
print(agents_path)

print("\nBestanden:")
for p in sorted(agents_path.iterdir()):
    print(p.name)

In [ ]:
from pathlib import Path
import langchain.agents

agents_path = Path(langchain.agents.__file__).parent

print("create_agent in factory.py:")
factory_text = (agents_path / "factory.py").read_text(encoding="utf-8")
print("create_agent" in factory_text)

print("\n__init__.py:")
init_text = (agents_path / "__init__.py").read_text(encoding="utf-8")
print(init_text)

In [ ]:
import sys

# Verwijder geladen oude LangChain agent-modules uit het geheugen
for name in list(sys.modules):
    if name == "langchain.agents" or name.startswith("langchain.agents."):
        del sys.modules[name]

from langchain.agents import create_agent

print("create_agent is beschikbaar:", create_agent)

In [ ]:
import importlib.metadata as md

for package in [
    "langchain",
    "langchain-core",
    "langgraph",
    "langchain-google-genai",
    "google-genai"
]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NIET GEVONDEN")

In [ ]:
from pathlib import Path
import langchain_core

base = Path(langchain_core.__file__).parent
tools_base = base / "tools" / "base.py"

print("LangChain Core:", langchain_core.__version__)
print("Bestand:", tools_base)
print("Bestaat:", tools_base.exists())

text = tools_base.read_text(encoding="utf-8")

print(
    "_DirectlyInjectedToolArg aanwezig:",
    "_DirectlyInjectedToolArg" in text
)

In [ ]:
import sys

for name in list(sys.modules):
    if (
        name == "langchain"
        or name.startswith("langchain.")
        or name == "langchain_core"
        or name.startswith("langchain_core.")
        or name == "langgraph"
        or name.startswith("langgraph.")
    ):
        del sys.modules[name]

import langchain
import langchain_core
import langgraph

print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)
print("LangGraph:", langgraph.__version__)

In [ ]:
import importlib.metadata as md

print("LangChain:", md.version("langchain"))
print("LangChain Core:", md.version("langchain-core"))
print("LangGraph:", md.version("langgraph"))

from langchain.agents import create_agent

print("create_agent is beschikbaar:", create_agent)

In [ ]:
from langchain.agents import create_agent

tools = [
    get_requirements,
    generate_test_cases
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a software testing assistant.

When the user asks for BDD/Gherkin test cases:
1. First use get_requirements to retrieve the functional requirements.
2. Use those requirements as the source of truth.
3. Then use generate_test_cases to generate the test cases.
4. Do not invent requirements, business rules, validation rules, limits,
   error messages or other behavior that is not present in the requirements.
"""
)

print("Agent aangemaakt.")

In [ ]:
from langchain_core.tools import tool

@tool
def get_requirements() -> str:
    """
    Retrieve the functional requirements for the password reset feature.
    """
    return """
Password Reset Requirements:

1. Only registered users can request a password reset.
2. The user must provide a valid email address.
3. A password reset link is sent to the registered email address.
4. The reset link can only be used once.
5. The user must enter the new password twice.
6. The two passwords must match.
7. The new password must comply with the application's password policy.
"""


@tool
def generate_test_cases(user_story: str) -> str:
    """
    Generate BDD/Gherkin test cases for a given user story.
    """

    prompt = f"""
You are a senior software tester.

Generate BDD test cases in Gherkin format for the following user story.

User story:
{user_story}

Create at least 10 different test scenarios.
Cover positive, negative, boundary and validation cases.

Use this structure:

Feature:
Scenario:
  Given
  When
  Then
"""

    response = llm.invoke(prompt)
    return response.content


print(type(get_requirements))
print(type(generate_test_cases))

In [ ]:
from langchain.agents import create_agent

tools = [
    get_requirements,
    generate_test_cases
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a software testing assistant.

When the user asks for BDD/Gherkin test cases:

1. First use get_requirements to retrieve the functional requirements.
2. Treat those requirements as the source of truth.
3. Then use generate_test_cases to generate the test cases.
4. Do not invent requirements, business rules, validation rules,
   limits, error messages or other behavior that is not present
   in the retrieved requirements.
"""
)

print("Agent aangemaakt:", agent)


In [ ]:
request = """
Create BDD test cases for this user story:

As a registered user,
I want to reset my password using my email address,
so that I can regain access to my account if I forget my password.
"""

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": request
        }
    ]
})

print(result["messages"][-1].content)

In [ ]:
print("get_requirements:")
print("  type:", type(get_requirements))
print("  schema:", get_requirements.args_schema)
print("  schema module:", get_requirements.args_schema.__module__)

print()

print("generate_test_cases:")
print("  type:", type(generate_test_cases))
print("  schema:", generate_test_cases.args_schema)
print("  schema module:", generate_test_cases.args_schema.__module__)

In [ ]:
from langchain_core.tools import tool

@tool
def generate_test_cases(user_story: str) -> str:
    """
    Prepare the user story for BDD/Gherkin test case generation.
    The agent itself generates the final test cases.
    """
    return f"""
Generate BDD/Gherkin test cases for this user story:

{user_story}

Use the functional requirements retrieved by get_requirements
as the source of truth.

Do not invent requirements, business rules, limits,
error messages or other behavior that is not specified.
"""

In [ ]:
tools = [
    get_requirements,
    generate_test_cases
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a software testing assistant.

When the user asks for BDD/Gherkin test cases:

1. First use get_requirements.
2. Treat the retrieved requirements as the source of truth.
3. Then use generate_test_cases.
4. Generate the final BDD/Gherkin test cases yourself.
5. Do not invent requirements, business rules, validation rules,
   limits, error messages or other behavior not present in the requirements.
"""
)

print("Agent opnieuw aangemaakt.")

In [ ]:
request = """
Create BDD test cases for this user story:

As a registered user,
I want to reset my password using my email address,
so that I can regain access to my account if I forget my password.
"""

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": request
        }
    ]
})

print(result["messages"][-1].content)

In [ ]:
import sys

for name in list(sys.modules):
    if (
        name == "langchain_google_genai"
        or name.startswith("langchain_google_genai.")
        or name == "langchain"
        or name.startswith("langchain.")
        or name == "langchain_core"
        or name.startswith("langchain_core.")
        or name == "langgraph"
        or name.startswith("langgraph.")
    ):
        del sys.modules[name]

print("Oude LangChain- en Gemini-modules verwijderd.")

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(
    r"C:\Users\Mill\agentsforQA\.env",
    override=True
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

print("LLM opnieuw geladen.")

In [ ]:
from langchain_core.tools import tool


@tool
def get_requirements() -> str:
    """
    Retrieve the functional requirements for the password reset feature.
    """
    return """
Password Reset Requirements:

1. Only registered users can request a password reset.
2. The user must provide a valid email address.
3. A password reset link is sent to the registered email address.
4. The reset link can only be used once.
5. The user must enter the new password twice.
6. The two passwords must match.
7. The new password must comply with the application's password policy.
"""


@tool
def generate_test_cases(user_story: str) -> str:
    """
    Prepare the user story for BDD/Gherkin test case generation.
    The agent itself generates the final test cases.
    """
    return f"""
Generate BDD/Gherkin test cases for this user story:

{user_story}

Use the functional requirements retrieved by get_requirements
as the source of truth.

Do not invent requirements, business rules, limits,
error messages or other behavior that is not specified.
"""


print("Tools opnieuw aangemaakt.")
print(type(get_requirements))
print(type(generate_test_cases))

In [ ]:
from langchain.agents import create_agent

tools = [
    get_requirements,
    generate_test_cases
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a software testing assistant.

When the user asks for BDD/Gherkin test cases:

1. First use get_requirements.
2. Treat the retrieved requirements as the source of truth.
3. Then use generate_test_cases.
4. Generate the final BDD/Gherkin test cases yourself.
5. Do not invent requirements, business rules, validation rules,
   limits, error messages or other behavior not present in the requirements.
"""
)

print("Agent opnieuw aangemaakt.")
print(type(agent))

In [ ]:
request = """
Generate BDD/Gherkin test cases for the password reset feature.
Use the available functional requirements as the source of truth.
"""

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": request
        }
    ]
})

print(result["messages"][-1].content)

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": request
        }
    ]
})

print(result["messages"][-1].content)

In [ ]:
requirements = get_requirements.invoke({})

print(requirements)

In [ ]:
result = generate_test_cases.invoke({
    "user_story": """
As a registered user,
I want to reset my password via email,
so that I can regain access to my account.
"""
})

print(result)

In [ ]:
requirements = get_requirements.invoke({})

prompt = generate_test_cases.invoke({
    "user_story": """
As a registered user,
I want to reset my password via email,
so that I can regain access to my account.
"""
})

print("=== REQUIREMENTS ===")
print(requirements)

print("\n=== TEST CASE GENERATION PROMPT ===")
print(prompt)

Stap 1 — De PDF/document reader toevoegen

We beginnen met de code die de documenten uit je Docs-map kan lezen. Dit is nodig voordat de agent requirements uit een PDF kan halen.

Waarom deze stap

Je agent moet eerst toegang krijgen tot de inhoud van de externe documenten. In plaats van requirements handmatig aan de tool mee te geven, lezen we ze rechtstreeks uit bestanden in de Docs-map.

Voeg eerst de benodigde imports toe aan je bestaande code:

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

Dat is duidelijk: je huidige .venv312 heeft langchain-community nog niet geïnstalleerd.
Stap 1 — langchain-community installeren
Waarom deze stap
PyPDFLoader zit niet in de standaard langchain package. De PDF-document loaders zitten in langchain-community. Zonder deze package kan Python de import niet vinden.
Omdat je in Jupyter werkt, installeren we hem rechtstreeks vanuit dezelfde Python-omgeving als je notebook.
Voer in een nieuwe notebookcel uit:

In [ ]:
%pip install -U langchain-community pypdf

De installatie is geslaagd. langchain-community en pypdf staan nu in je .venv312.
Er is wel één belangrijk punt: langchain-text-splitters is bijgewerkt van 0.3.11 naar 1.1.2. Dat is normaal als gevolg van de huidige langchain-community-versie, maar we moeten daarna controleren of je bestaande code nog werkt.
Stap 2 — Kernel opnieuw starten
Waarom deze stap
Jupyter had de oude Python-omgeving al geladen voordat we de packages installeerden. Door de kernel opnieuw te starten, krijgt je notebook toegang tot de zojuist geïnstalleerde packages.
In VS Code:
Kernel → Restart
Daarna voer je alleen dit uit:

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

print("PyPDFLoader werkt")

Goed. De import werkt. De melding is alleen een DeprecationWarning; voor deze les kunnen we doorgaan.
Stap 2 — De PDF-bestanden uit Docs inladen
Waarom deze stap
De tool moet straks de requirements uit externe PDF-bestanden kunnen lezen. Daarom zoeken we eerst alle .pdf-bestanden in de map Docs en laden we de inhoud van iedere PDF in documents.

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

docs_folder = Path("./Docs")

documents = []

for pdf_file in docs_folder.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_file))
    documents.extend(loader.load())

print(f"Aantal geladen pagina's: {len(documents)}")

Dat betekent dat de code werkt, maar dat Python in ./Docs geen PDF-bestanden heeft gevonden. We moeten eerst controleren waar je notebook daadwerkelijk draait en of daar een Docs-map staat.
Stap 2a — Controleren vanuit welke map Python werkt
Waarom deze stap
Path("./Docs") is relatief aan de huidige werkmap van je Jupyter-kernel. Als je notebook vanuit een andere map wordt gestart, kan ./Docs naar de verkeerde locatie wijzen.

In [ ]:
from pathlib import Path

print("Huidige werkmap:")
print(Path.cwd())

print("\nBestanden en mappen:")
for item in Path.cwd().iterdir():
    print(item.name)

Stap 2b — Zoeken naar de Docs-map

Waarom deze stap

We willen niet zomaar een nieuwe map maken als de PDF's al ergens in je project staan. Met deze stap zoeken we binnen agentsforQA naar bestaande mappen met de naam Docs.

In [ ]:
from pathlib import Path

project_folder = Path(r"C:\Users\Mill\agentsforQA")

docs_folders = list(project_folder.rglob("Docs"))

print("Gevonden Docs-mappen:")

for folder in docs_folders:
    print(folder)

Stap 2c — De juiste Docs-map gebruiken

Waarom deze stap

Onze huidige code gebruikt ./Docs, maar die map bestaat niet in Section 6. We wijzen daarom expliciet naar de bestaande Docs-map van Section 3. Zo kunnen we controleren of de PDF's daar daadwerkelijk worden ingelezen.

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

docs_folder = Path(
    r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
)

documents = []

for pdf_file in docs_folder.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_file))
    documents.extend(loader.load())

print(f"Aantal geladen pagina's: {len(documents)}")

Stap 2d — Inhoud van de juiste Docs-map bekijken

Waarom deze stap

We moeten eerst zien welke bestanden daar daadwerkelijk aanwezig zijn. Daarna kunnen we de loader precies op de aanwezige bestanden richten.

In [ ]:
from pathlib import Path

docs_folder = Path(
    r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
)

print("Inhoud van Docs:")

for item in docs_folder.iterdir():
    print(item.name)

Stap 2e — Alle PDF-bestanden binnen agentsforQA zoeken

Waarom deze stap

De eerdere zoekactie vond alleen mappen met de naam Docs. Nu zoeken we rechtstreeks naar alle PDF-bestanden binnen je project. Daarmee kunnen we de exacte locatie van de requirements-PDF's vinden.

In [ ]:
from pathlib import Path

project_folder = Path(r"C:\Users\Mill\agentsforQA")

pdf_files = list(project_folder.rglob("*.pdf"))

print(f"Aantal PDF-bestanden gevonden: {len(pdf_files)}")

for pdf_file in pdf_files:
    print(pdf_file)

Stap 2f — Controleren welke bestanden in Section 3 staan

Waarom deze stap

Section 3 heet WorkingWithExternalDocs en bevat wel een Docs-map. We willen controleren of de cursusbestanden misschien buiten Docs staan, bijvoorbeeld in de hoofdmap van Section 3.

In [ ]:
from pathlib import Path

section3 = Path(
    r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs"
)

for item in section3.rglob("*"):
    if item.is_file():
        print(item)

Stap 2g — Zoeken buiten agentsforQA

Waarom deze stap

Tot nu toe hebben we alleen binnen agentsforQA gezocht. Omdat de cursus-PDF's daar niet aanwezig zijn, zoeken we nu in je gebruikersmap C:\Users\Mill naar PDF's. We filteren daarbij de bekende Python- en virtual-environmentmappen eruit, zodat we niet opnieuw tientallen package-bestanden krijgen.

In [ ]:
from pathlib import Path

user_folder = Path(r"C:\Users\Mill")

pdf_files = []

for pdf_file in user_folder.rglob("*.pdf"):
    path_str = str(pdf_file)

    if "\\.venv\\" not in path_str and "\\AppData\\" not in path_str:
        pdf_files.append(pdf_file)

print(f"Aantal relevante PDF-bestanden gevonden: {len(pdf_files)}")

for pdf_file in pdf_files:
    print(pdf_file)

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

docs_folder = Path("./Docs")

documents = []

for pdf_file in docs_folder.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_file))
    documents.extend(loader.load())

print(f"Aantal geladen pagina's: {len(documents)}")

Stap 1 — Controleren of de PDF's aanwezig zijn

Waarom deze stap

We controleren eerst of de documenten daadwerkelijk in de juiste Docs-map staan. Zo voorkomen we dat we de volgende code uitvoeren terwijl de map nog leeg is.

In [ ]:
from pathlib import Path

docs_folder = Path(r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs")

pdf_files = list(docs_folder.glob("*.pdf"))

print(f"Aantal PDF-bestanden: {len(pdf_files)}")

for pdf_file in pdf_files:
    print(pdf_file.name)

Stap 2 — De PDF-documenten inladen

Waarom deze stap

PyPDFLoader leest iedere PDF uit de Docs-map en zet de inhoud per pagina om naar documenten die we later aan de AI-agent kunnen meegeven.

Gebruik nu:

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

docs_folder = Path(r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs")

documents = []

for pdf_file in docs_folder.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_file))
    documents.extend(loader.load())

print(f"Aantal geladen pagina's: {len(documents)}")

Stap 3 — Controleren welke informatie is ingelezen

Waarom deze stap

Elke pagina is nu een afzonderlijk Document. We controleren kort de inhoud voordat we de documenten samenvoegen tot één requirement_text, zoals in Lesson 63.

In [ ]:
print(documents[0].page_content[:2000])

Stap 4 — De inhoud van alle pagina's samenvoegen

Waarom deze stap

We hebben nu 65 afzonderlijke pagina-documenten. De agent moet straks niet alleen de eerste pagina gebruiken, maar de inhoud van alle ingelezen PDF-pagina's als één requirement_text kunnen verwerken.

Gebruik:

In [ ]:
requirement_text = "\n\n".join(
    document.page_content.strip()
    for document in documents
)

print(f"Aantal tekens in requirement_text: {len(requirement_text)}")

Stap 5 — De requirements beperken

Waarom deze stap

165.273 tekens is te veel om zonder beperking in de prompt te stoppen. In de les wordt daarom de requirement_text ingekort voordat deze aan de test-case-generator wordt doorgegeven.

In [ ]:
requirement_text = requirement_text[:10000]

print(f"Aantal tekens na beperking: {len(requirement_text)}")

Stap 6 — Controleren wat de agent straks als requirements krijgt

Waarom deze stap

Voordat we de tekst in de prompt plaatsen, controleren we kort of de ingekorte tekst nog steeds leesbare inhoud bevat. Dit is alleen een controle; we wijzigen de variabele niet.

In [ ]:
print(requirement_text[:2000])

Stap 7 — Prompt aanpassen zodat deze de PDF-inhoud gebruikt

Waarom deze stap

De oorspronkelijke generator werkte met een requirement die rechtstreeks als argument werd meegegeven. In deze les komt de requirement nu uit de PDF's. Daarom moet de prompt de variabele requirement_text gebruiken.

Maak eerst de prompt aan:

In [ ]:
prompt = f"""
Read the following requirements and generate at least 10 test scenarios
in BDD format.

Requirements:
{requirement_text}
"""

print(prompt[:3000])

Stap 8 — Bestaande generator erbij pakken

Waarom deze stap

Lesson 63 verandert niet alleen de prompt. De generator moet uiteindelijk de requirements uit de PDF gebruiken in plaats van een requirement die handmatig als argument wordt meegegeven. Daarom moeten we eerst jouw bestaande generator zien voordat we iets aanpassen.

Zoek in je notebook naar de code waarin je eerder iets hebt zoals:

Stap 8 — user_story verwijderen uit de tool

Waarom deze stap

Je huidige functie:

In [ ]:
@tool
def generate_test_cases() -> str:
    """
    Read the requirements from the PDF and generate at least
    10 BDD/Gherkin test scenarios.
    """

    return f"""
Generate at least 10 BDD/Gherkin test scenarios based on the
following requirements:

{requirement_text}

Use only the information provided in the requirements.

Do not invent requirements, business rules, limits, error messages,
or other behavior that is not specified.
"""

In [ ]:
print("Tool opnieuw aangemaakt.")
print(type(get_requirements))
print(type(generate_test_cases))

Stap 9 — De agentconfiguratie controleren

Waarom deze stap

We hebben generate_test_cases() aangepast zodat deze geen argument meer nodig heeft. Als de agent nog een prompt heeft waarin staat dat hij een user_story aan generate_test_cases moet meegeven, ontstaat er straks een mismatch.

Zoek in je notebook naar het stuk waarin je de agent aanmaakt. Dat kan er bijvoorbeeld uitzien als:

Stap 9 — De system prompt aanpassen

Waarom deze stap

De requirements komen nu rechtstreeks uit de PDF's. Daarom hoeft de agent niet eerst get_requirements aan te roepen voordat hij generate_test_cases gebruikt.

Vervang alleen de system_prompt door deze versie:

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a software testing assistant.

When the user asks for BDD/Gherkin test cases:

1. Use generate_test_cases to generate the test scenarios from the requirements provided by the tool.
2. Treat the requirements as the source of truth.
3. Generate at least 10 BDD/Gherkin test scenarios.
4. Do not invent requirements, business rules, validation rules,
   limits, error messages or other behavior not present in the requirements.
"""
)

print("Agent opnieuw aangemaakt.")
print(type(agent))

Stap 10 — De agent testen

Waarom deze stap

Tot nu toe hebben we alleen de onderdelen afzonderlijk aangepast. Met deze test controleren we of de volledige keten werkt:

PDF's → requirement_text → generate_test_cases() → agent → BDD/Gherkin-testscenario's.

Gebruik een eenvoudige opdracht:

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Generate at least 10 BDD/Gherkin test scenarios based on the requirements."
        }
    ]
})

print(response)

Stap 11 — De tools-lijst opnieuw opbouwen

Waarom deze stap

De agent krijgt zijn beschikbare tools via:

In [ ]:
tools=tools

Dit is de code voor het aanmaken van de agent. Omdat je nu bezig bent met Lesson 63, zou ik deze stap eerst uitvoeren zonder iets aan de code te veranderen.

Waarom deze stap

create_agent() maakt de agent aan met het model (llm) en de tools die op dat moment in tools staan. Belangrijk is dus dat de tools-lijst al de juiste versie van generate_test_cases bevat voordat je deze code uitvoert.

In [ ]:
try:
    from langchain.agents import create_agent

except ModuleNotFoundError:
    %pip install "langchain>=1.0,<2.0"
    from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools
)

print(agent)

In [ ]:
print("Aantal tools:", len(tools))

for tool in tools:
    print(tool.name)

Dit bevestigt het probleem. Er staan twee tools in de agent:

get_requirements
generate_test_cases

Maar we moeten nu controleren of generate_test_cases in tools de nieuwe versie is die je zojuist hebt aangemaakt. Alleen de naam zegt dat nog niet.

Stap 12 — Controleren welke generate_test_cases in tools staat
Waarom deze stap
We hebben eerder een nieuwe generate_test_cases() aangemaakt zonder user_story. De agent gebruikte echter nog de oude versie mét user_story. Met onderstaande controle zien we de daadwerkelijke beschrijving van de tool die in tools zit.
Voer dit uit:

In [ ]:
for tool in tools:
    print(f"\nTool: {tool.name}")
    print(tool.description)

Stap 13 — De nieuwe generate_test_cases aan tools toevoegen

Waarom deze stap

De nieuwe functie bestaat al, maar tools verwijst nog naar het oude StructuredTool-object. Door de lijst opnieuw op te bouwen, krijgt de agent de nieuwe versie met requirement_text en zonder user_story.

Voer dit uit:

In [ ]:
tools = [
    get_requirements,
    generate_test_cases
]

print("Tools opnieuw opgebouwd.")

for tool in tools:
    print(f"\nTool: {tool.name}")
    print(tool.description)

Stap 14 — Agent opnieuw aanmaken

Waarom deze stap

De agent die eerder is aangemaakt, bevat nog de oude tooldefinities. Omdat tools nu is bijgewerkt, moeten we de agent opnieuw aanmaken zodat hij de nieuwe generate_test_cases gebruikt.

Voer nu dit uit:

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools
)

print("Agent opnieuw aangemaakt.")
print(type(agent))

Stap 15 — De agent testen

Waarom deze stap

We willen nu controleren of de agent daadwerkelijk de nieuwe PDF-tool gebruikt en niet opnieuw de oude Password Reset-flow volgt.

Voer deze test uit:

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Generate at least 10 BDD/Gherkin test scenarios based on the requirements."
        }
    ]
})

print(response)

Stap 16 — Alleen het resultaat van de nieuwe tool bekijken

Je volledige response bevat enorm veel interne informatie en Gemini-signatures. We hoeven die niet allemaal te printen. We willen alleen zien wat de agent uiteindelijk heeft gegenereerd.

Waarom deze stap

Door alleen de laatste AI-boodschap te tonen, kunnen we controleren of de PDF-inhoud daadwerkelijk is gebruikt en of er minstens 10 BDD/Gherkin-scenario's zijn gegenereerd.

Voer dit uit:

In [ ]:
final_message = response["messages"][-1]

print(final_message.content)

Dit is goed. De nieuwe tool wordt daadwerkelijk gebruikt en Gemini genereert nu de BDD/Gherkin-scenario’s op basis van de requirements.

Belangrijk is dat de output begint met Password Reset. Dat betekent dat de agent nog steeds de oude requirements gebruikt in de uiteindelijke prompt, en niet de drie Netflix-PDF’s die we eerder hebben ingelezen.

Voor de les is de volgende stap daarom niet om nu allerlei code te wijzigen. Eerst moeten we controleren of de volledige output inderdaad uit de tool komt en hoeveel scenario’s Gemini heeft gegenereerd.


Stap 17 — Volledige gegenereerde output controleren

Waarom deze stap

We willen controleren of Gemini daadwerkelijk minimaal 10 scenario’s heeft gegenereerd en of de output inhoudelijk overeenkomt met de requirements die de tool gebruikt.

In [ ]:
final_message = response["messages"][-1]

print(final_message.content)

Stap 18 — get_requirements uit de tools halen

Waarom deze stap

De nieuwe lesopzet moet de requirements rechtstreeks uit de PDF lezen. get_requirements levert nog de oude Password Reset-requirements en veroorzaakt daardoor de vermenging. We halen deze oude tool daarom uit tools.

Voer eerst dit uit:

In [ ]:
tools = [
    generate_test_cases
]

print("Aantal tools:", len(tools))

for tool in tools:
    print(f"\nTool: {tool.name}")
    print(tool.description)

Stap 19 — Agent opnieuw aanmaken

Waarom deze stap

De bestaande agent is eerder aangemaakt met beide tools. Alleen de variabele tools aanpassen is niet voldoende; we moeten de agent opnieuw opbouwen zodat hij daadwerkelijk alleen generate_test_cases kent.

Voer nu uit:

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools
)

print("Agent opnieuw aangemaakt.")
print(type(agent))

Stap 20 — De agent opnieuw testen

Waarom deze stap

We willen nu controleren of de agent uitsluitend generate_test_cases aanroept en geen Password Reset-requirements meer binnenhaalt. De prompt hoeft geen requirements mee te geven, omdat de tool die zelf uit de PDF heeft gelezen.

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Generate at least 10 BDD/Gherkin test scenarios based on the requirements."
        }
    ]
})

print(response)

Stap 21 — Alleen het uiteindelijke antwoord bekijken

Waarom deze stap

De volledige response bevat veel technische informatie: HumanMessage, AIMessage, ToolMessage, Gemini-signatures en tokeninformatie. Voor de gebruiker willen we uiteindelijk alleen het gegenereerde BDD-resultaat zien.

Gebruik daarom:

In [ ]:
final_message = response["messages"][-1]

print(final_message.content)

Stap 1 — Controleer je huidige project

Waarom deze stap

We willen eerst controleren dat we verdergaan vanuit de code die we in les 62 hebben gemaakt, dus met Google Gemini als LLM. We willen niet per ongeluk teruggaan naar de lokale Ollama-LLM.

Ga naar:

Stap 1 — vervang je huidige functie
Waarom deze stap
Lesson 63 draait om deze verandering:
user story → BDD
wordt:
PDF/document → requirements → BDD
Daarom halen we user_story uit de functie en lezen we de documenten binnen de tool.
Gebruik voorlopig deze versie:

In [ ]:
@tool
def generate_test_cases() -> str:
    """Read requirements from PDF documents and generate at least ten BDD/Gherkin test scenarios."""

    # Read requirements from the documents
    documents = load_documents("./Docs")

    requirement_text = "\n\n".join(
        f"File: {doc.metadata.get('source', 'unknown')}\n{doc.page_content.strip()}"
        for doc in documents
    )

    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Your task is to generate at least ten test scenarios in BDD/Gherkin format
based only on the following requirements.

Requirements:
{requirements}

Use only the information provided in the requirements.

Do not invent requirements, business rules, limits, error messages,
or other behavior that is not specified.

Use the following BDD format:

Feature: [feature name]

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]
""")

    prompt_value = prompt.invoke({
        "requirements": requirement_text
    })

    response = llm.invoke(prompt_value)

    return response.content

In [ ]:
print(type(tool))
print(tool)

In [ ]:
from langchain_core.tools import tool

In [ ]:
print(type(tool))

In [ ]:
print(type(tool))

In [ ]:
@tool
def generate_test_cases() -> str:
    """Read the requirements from the PDF and generate at least 10 BDD/Gherkin test scenarios."""

    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Your task is to generate at least ten test scenarios in BDD/Gherkin format
based only on the following requirements.

Requirements:
{requirements}

Use only the information provided in the requirements.

Do not invent requirements, business rules, limits, error messages,
or other behavior that is not specified.

Use the following BDD format:

Feature: [feature name]

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]
""")

    prompt_value = prompt.invoke({
        "requirements": requirement_text
    })

    response = llm.invoke(prompt_value)

    return response.content

In [ ]:
print(type(requirement_text))

In [ ]:
print(type(requirement_text))

generate_test_cases aanpassen

Waarom deze stap

Je huidige functie verwacht nog een user_story. In Lesson 63 moet de functie juist de bestaande requirement_text gebruiken, die al uit de PDF is gehaald.

Vervang je huidige generate_test_cases volledig door:

In [ ]:
@tool
def generate_test_cases() -> str:
    """Read the requirements from the PDF and generate at least 10 BDD/Gherkin test scenarios."""

    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Your task is to generate at least ten test scenarios in BDD/Gherkin format
based only on the following requirements.

Requirements:
{requirements}

Use only the information provided in the requirements.

Do not invent requirements, business rules, limits, error messages,
or other behavior that is not specified.

Use the following BDD format:

Feature: [feature name]

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]
""")

    prompt_value = prompt.invoke({
        "requirements": requirement_text
    })

    response = llm.invoke(prompt_value)

    return response.content

Stap 4 — controleren of de tool correct is aangemaakt

Waarom deze stap

We controleren eerst of generate_test_cases nu inderdaad een LangChain-tool is en geen gewone Python-functie. Daarna kunnen we hem daadwerkelijk uitvoeren.

Voer uit:

In [ ]:
print(type(generate_test_cases))
print(generate_test_cases.name)
print(generate_test_cases.description)

Stap 5 — de tool zelfstandig testen

Waarom deze stap

We testen nu of de tool daadwerkelijk requirement_text uit de PDF gebruikt en vervolgens Gemini aanroept om de BDD-scenario's te genereren.

Voer uit:

In [ ]:
result = generate_test_cases.invoke({})
print(result)

De fout is duidelijk: PromptTemplate is op dit moment niet beschikbaar in de notebook-kernel. De functie zelf is dus goed, maar de benodigde import ontbreekt.

Stap 6 — PromptTemplate importeren

Waarom deze stap

Je functie gebruikt PromptTemplate.from_template(...). Python moet daarom eerst weten waar PromptTemplate vandaan komt.

Voer alleen dit uit:

In [ ]:
from langchain_core.prompts import PromptTemplate

Stap 7 — de tool opnieuw uitvoeren

Waarom deze stap

Nu PromptTemplate correct is geïmporteerd, kunnen we dezelfde test opnieuw uitvoeren. We verwachten nu dat de tool de bestaande requirement_text gebruikt en Gemini aanroept.

Voer uit:

In [ ]:
result = generate_test_cases.invoke({})
print(result)

Dit is geen foutmelding. Het is een waarschuwing van de Gemini-library over Automatic Function Calling (AFC).

Belangrijker: je krijgt blijkbaar nog geen BDD-output te zien. We controleren daarom eerst of result daadwerkelijk gevuld is.

Stap 8 — resultaat controleren

Waarom deze stap

We willen onderscheid maken tussen een waarschuwing en een daadwerkelijk resultaat. Voer alleen dit uit:

In [ ]:
print(type(result))
print(len(result))

Stap 9 — inhoud van het resultaat bekijken

Waarom deze stap

We willen nu zien wat Gemini precies heeft teruggegeven. Omdat result een lijst is, bekijken we het eerste item.

Voer uit:

In [ ]:
print(result[0])

In [ ]:
generate_test_cases

In [ ]:
@tool
def generate_test_cases() -> str:
    """Read the requirements from the PDF and generate at least 10 BDD/Gherkin test scenarios."""

    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Your task is to generate at least ten test scenarios in BDD/Gherkin format
based only on the following requirements.

Requirements:
{requirements}

Use only the information provided in the requirements.

Do not invent requirements, business rules, limits, error messages,
or other behavior that is not specified.

Use the following BDD format:

Feature: [feature name]

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]
""")

    prompt_value = prompt.invoke({
        "requirements": requirement_text
    })

    response = llm.invoke(prompt_value)

    return response.content

In [ ]:
print(type(generate_test_cases))
print(generate_test_cases.name)
print(generate_test_cases.description)
print(generate_test_cases.args_schema.model_json_schema())

In [ ]:
result = generate_test_cases.invoke({})
print(result)

In [ ]:
print(llm)

In [ ]:
ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

In [ ]:
from google import genai

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

for model in client.models.list():
    print(model.name)

In [ ]:
from dotenv import load_dotenv
import os

from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(r"C:\Users\Mill\agentsforQA\.env", override=True)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

In [ ]:
print(llm)

In [ ]:
result = generate_test_cases.invoke({})
print(result)

In [ ]:
from dotenv import load_dotenv
import os

from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(r"C:\Users\Mill\agentsforQA\.env", override=True)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

print(llm)

In [ ]:
result = generate_test_cases.invoke({})
print(result)

In [ ]:
generate_test_cases.invoke({})

In [ ]:
result = generate_test_cases.invoke({})
print(result)

Ja. Je hebt gelijk. Je hebt mij zojuist de les gegeven, dus we moeten niet terug naar les 63.

Op basis van de transcript die je net gaf, is dit wat je nu moet doen:

De huidige les verandert de manier waarop je de tool aanroept. Je hoeft geen user_story meer mee te geven. De tool leest zelf de requirements uit de PDF en gebruikt die om de BDD-testcases te genereren.

De flow is nu:

PDF → requirements lezen → requirement_text → Gemini → minimaal 10 BDD-testcases

Dus je volgende stap in je notebook/code is:

Waarom deze stap

We verwijderen de handmatig meegegeven user_story, omdat de tool de requirements inmiddels zelf uit de PDF haalt. Daardoor hoeft de gebruiker alleen de tool aan te roepen.

In [ ]:
response = generate_test_cases.invoke({})
print(response)

In [ ]:
print(llm)

In [ ]:
response = generate_test_cases.invoke({})
print(response)

In [ ]:
response = generate_test_cases.invoke({})
print(response)

In [ ]:
from langchain_ollama import ChatOllama

from langchain.agents import create_agent

llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=tools
)

print(agent)

In [ ]:
%pip install -U langchain-ollama


In [ ]:
from langchain_ollama import ChatOllama

print("ChatOllama import OK")

In [ ]:
from langchain.agents import create_agent

llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=tools
)

print(agent)

In [ ]:
tools=tools

In [ ]:
tools = [generate_test_cases]

print(tools)

In [ ]:
print(generate_test_cases)

In [ ]:
tools = [generate_test_cases]

print(tools)

In [ ]:
from langchain.agents import create_agent

llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=tools
)

print(agent)

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

In [ ]:
print(ChatOllama)

In [ ]:
llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

print(llm)

In [ ]:
tools = [generate_test_cases]

In [ ]:
tools = [generate_test_cases]

print(tools)

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools
)

print(agent)

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Generate BDD test cases for this user story: As a registered user, I want to reset my password so that I can regain access to my account."
        }
    ]
})

print(response)

In [ ]:
@tool
def generate_test_cases() -> str:
    """Read requirements from PDF documents and generate at least ten BDD/Gherkin test scenarios."""

    # Read requirements from the documents
    documents = load_documents("./Docs")

    requirement_text = "\n\n".join(
        f"File: {doc.metadata.get('source', 'unknown')}\n{doc.page_content.strip()}"
        for doc in documents
    )

    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Your task is to generate at least ten test scenarios in BDD/Gherkin format
based only on the following requirements.

Requirements:
{requirements}

Use only the information provided in the requirements.

Do not invent requirements, business rules, limits, error messages,
or other behavior that is not specified.

Use the following BDD format:

Feature: [feature name]

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]
""")

    prompt_value = prompt.invoke({
        "requirements": requirement_text
    })

    response = llm.invoke(prompt_value)

    return response.content

In [ ]:
tools = [generate_test_cases]

print(tools)

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools
)

print(agent)

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def load_documents(folder_path: str):
    documents = []

    for pdf_file in Path(folder_path).glob("*.pdf"):
        loader = PyPDFLoader(str(pdf_file))
        documents.extend(loader.load())

    return documents

In [ ]:
documents = load_documents("./Docs")

print(f"Aantal geladen pagina's: {len(documents)}")

In [1]:
from pathlib import Path

docs_path = Path("./Docs")

print("Huidige werkmap:")
print(Path.cwd())

print("\nDocs-pad:")
print(docs_path.resolve())

print("\nBestanden in Docs:")
for file in docs_path.glob("*"):
    print(file)

Huidige werkmap:
c:\Users\Mill\agentsforQA\.venv\Section6_PlaywrightToolkit

Docs-pad:
C:\Users\Mill\agentsforQA\.venv\Section6_PlaywrightToolkit\Docs

Bestanden in Docs:


In [9]:
from pathlib import Path

project_path = Path(r"C:\Users\Mill\agentsforQA")

pdf_files = list(project_path.rglob("*.pdf"))

print("Gevonden PDF-bestanden:")
for pdf in pdf_files:
    print(pdf)

Gevonden PDF-bestanden:
C:\Users\Mill\agentsforQA\.venv\Lib\site-packages\matplotlib\mpl-data\images\back.pdf
C:\Users\Mill\agentsforQA\.venv\Lib\site-packages\matplotlib\mpl-data\images\filesave.pdf
C:\Users\Mill\agentsforQA\.venv\Lib\site-packages\matplotlib\mpl-data\images\forward.pdf
C:\Users\Mill\agentsforQA\.venv\Lib\site-packages\matplotlib\mpl-data\images\hand.pdf
C:\Users\Mill\agentsforQA\.venv\Lib\site-packages\matplotlib\mpl-data\images\help.pdf
C:\Users\Mill\agentsforQA\.venv\Lib\site-packages\matplotlib\mpl-data\images\home.pdf
C:\Users\Mill\agentsforQA\.venv\Lib\site-packages\matplotlib\mpl-data\images\matplotlib.pdf
C:\Users\Mill\agentsforQA\.venv\Lib\site-packages\matplotlib\mpl-data\images\move.pdf
C:\Users\Mill\agentsforQA\.venv\Lib\site-packages\matplotlib\mpl-data\images\qt4_editor_options.pdf
C:\Users\Mill\agentsforQA\.venv\Lib\site-packages\matplotlib\mpl-data\images\subplots.pdf
C:\Users\Mill\agentsforQA\.venv\Lib\site-packages\matplotlib\mpl-data\images\zoom_to_

In [10]:
documents = load_documents(
    r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
)

print(f"Aantal geladen pagina's: {len(documents)}")

NameError: name 'load_documents' is not defined

In [13]:
documents = load_documents("./Docs")

NameError: name 'load_documents' is not defined

In [5]:
documents = load_documents(
    r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
)

NameError: name 'load_documents' is not defined

In [6]:
@tool
def generate_test_cases() -> str:
    """Read requirements from PDF documents and generate at least ten BDD/Gherkin test scenarios."""

    # Read requirements from the documents
    documents = load_documents(
        r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
    )

    requirement_text = "\n\n".join(
        f"File: {doc.metadata.get('source', 'unknown')}\n{doc.page_content.strip()}"
        for doc in documents
    )

    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Your task is to generate at least ten test scenarios in BDD/Gherkin format
based only on the following requirements.

Requirements:
{requirements}

Use only the information provided in the requirements.

Do not invent requirements, business rules, limits, error messages,
or other behavior that is not specified.

Use the following BDD format:

Feature: [feature name]

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]
""")

    prompt_value = prompt.invoke({
        "requirements": requirement_text
    })

    response = llm.invoke(prompt_value)

    return response.content

NameError: name 'tool' is not defined

In [ ]:
tools = [generate_test_cases]

agent = create_agent(
    model=llm,
    tools=tools
)

NameError: name 'generate_test_cases' is not defined

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Generate at least 10 BDD/Gherkin test scenarios based on the requirements in the PDF documents."
        }
    ]
})

print(response)

NameError: name 'agent' is not defined

In [ ]:
from langchain_core.prompts import PromptTemplate

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Generate at least 10 BDD/Gherkin test scenarios based on the requirements in the PDF documents."
        }
    ]
})

print(response)

NameError: name 'agent' is not defined

Dit ziet er goed uit. Je hebt de oude user_story-input al verwijderd. Je invocation past dus bij de nieuwe les.

Stap 2 — Controleer eerst de tool zelf

Je hebt nu:

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Generate at least 10 BDD/Gherkin test scenarios based on the requirements in the PDF documents."
        }
    ]
})

print(response)

NameError: name 'agent' is not defined

Stap 3 — Controleer nu je tool

We moeten nu zien hoe generate_test_cases is gedefinieerd. Dit is belangrijk omdat jouw resultaat maar 7 scenario's bevat, terwijl je prompt vraagt om minimaal 10.

Je tool heeft kennelijk deze naam:

Goed. Nu zien we precies waar het probleem zit.

Je tool is correct opgebouwd en de PDF wordt daadwerkelijk gelezen. Alleen is er een kleine inconsistentie: je prompt vraagt om minimaal 10 scenario's, maar Qwen heeft er uiteindelijk 7 gegenereerd.

We gaan daarom eerst niets aan de architectuur veranderen. We gaan de tool iets duidelijker instrueren.

Stap 4 — Pas de prompt aan

Je hebt nu:

Vervang dit gedeelte door:

Waarom deze stap

We willen dat het model niet alleen "at least ten" ziet, maar expliciet controleert dat het antwoord minimaal 10 afzonderlijke Scenario's bevat. Dit is vooral nuttig bij een lokaal model zoals Qwen 2.5 7B, dat soms minder items genereert dan gevraagd.

Gebruik:

In [17]:
print(generate_test_cases)

name='generate_test_cases' description='Generate BDD test cases from a user story.' args_schema=<class 'langchain_core.utils.pydantic.generate_test_cases'> func=<function generate_test_cases at 0x000001565DD5DDA0>


In [20]:
@tool
def generate_test_cases() -> str:
    """Read requirements from PDF documents and generate at least ten BDD/Gherkin test scenarios."""

    documents = load_documents(
        r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
    )

    requirement_text = "\n\n".join(
        f"File: {doc.metadata.get('source', 'unknown')}\n{doc.page_content.strip()}"
        for doc in documents
    )

    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Your task is to generate at least 10 separate test scenarios in BDD/Gherkin format
based only on the following requirements.

You MUST generate at least 10 scenarios.
Do not stop before you have generated 10 scenarios.

Requirements:
{requirements}

Use only the information provided in the requirements.

Do not invent requirements, business rules, limits, error messages,
or other behavior that is not specified.

Use the following BDD format:

Feature: [feature name]

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]
""")

    prompt_value = prompt.invoke({
        "requirements": requirement_text
    })

    response = llm.invoke(prompt_value)

    return response.content

In [21]:
print(generate_test_cases)

name='generate_test_cases' description='Read requirements from PDF documents and generate at least ten BDD/Gherkin test scenarios.' args_schema=<class 'langchain_core.utils.pydantic.generate_test_cases'> func=<function generate_test_cases at 0x000001565DD5E840>


In [25]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=[generate_test_cases]
)

print(agent)

In [28]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Generate BDD/Gherkin test scenarios based on the requirements in the PDF documents."
        }
    ]
})

print(result["messages"][-1].content)

NameError: name 'load_documents' is not defined

In [29]:
print(load_documents)

NameError: name 'load_documents' is not defined

In [30]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

def load_documents(folder_path):
    loader = PyPDFDirectoryLoader(folder_path)
    return loader.load()

print("load_documents is beschikbaar")

load_documents is beschikbaar


C:\Users\Mill\AppData\Local\Temp\ipykernel_23848\2821652495.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [31]:
docs = load_documents(
    r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
)

print(f"Aantal geladen documenten/pagina's: {len(docs)}")

Aantal geladen documenten/pagina's: 65


In [32]:
docs = load_documents(
    r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
)

print(f"Aantal geladen documenten/pagina's: {len(docs)}")

Aantal geladen documenten/pagina's: 65


In [33]:
test_output = generate_test_cases.invoke({})

print(test_output)

Feature: Netflix Distributed System Design

Scenario: Ensuring High Availability for Streaming Services
  Given the system is designed with microservices architecture
  When a request arrives at the endpoint
  Then the system calls other micro-services for required data

Scenario: Handling Network Failures and System Outages
  Given the system uses Hystrix for resilience
  When a cascading failure occurs
  Then the system stops the error from propagating to other services

Scenario: Minimizing Streaming Latency
  Given the system uses Hystrix commands to control wait times
  When the network connection is unreliable
  Then the client switches to nearby servers

Scenario: Supporting Scalability
  Given the system uses AWS Auto Scaling Service
  When the volume of requests increases
  Then the system automatically launches more elastic instances

Scenario: Using Custom Caching Layer
  Given the system uses EV-cache for faster access
  When an instance failure occurs
  Then the data can b

In [34]:
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate


@tool
def generate_test_cases() -> str:
    """Read requirements from PDF documents and generate at least ten BDD/Gherkin test scenarios."""

    # Load the PDF documents
    documents = load_documents(
        r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
    )

    # Combine the content of all PDF pages
    requirement_text = "\n\n".join(
        f"File: {doc.metadata.get('source', 'unknown')}\n"
        f"{doc.page_content.strip()}"
        for doc in documents
    )

    # Prompt for the LLM
    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Your task is to generate BDD/Gherkin test scenarios based only on the
requirements provided below.

IMPORTANT:
- Generate AT LEAST 10 separate scenarios.
- You MUST provide 10 or more scenarios.
- Do NOT stop after 7, 8, or 9 scenarios.
- Count the scenarios before finishing your response.
- If fewer than 10 scenarios have been generated, continue generating
  additional scenarios based on other requirements from the documents.
- Every scenario must test a distinct behavior, requirement, condition,
  or system characteristic.
- Do not invent requirements or behavior that are not supported by the
  provided documents.

Requirements:
{requirements}

Use this BDD/Gherkin format:

Feature: [feature name]

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]

Return only the BDD/Gherkin test scenarios.
""")

    # Insert the requirements into the prompt
    prompt_value = prompt.invoke({
        "requirements": requirement_text
    })

    # Generate the test scenarios with the LLM
    response = llm.invoke(prompt_value)

    return response.content


print(generate_test_cases)

name='generate_test_cases' description='Read requirements from PDF documents and generate at least ten BDD/Gherkin test scenarios.' args_schema=<class 'langchain_core.utils.pydantic.generate_test_cases'> func=<function generate_test_cases at 0x0000015661EF6700>


In [35]:
test_output = generate_test_cases.invoke({})

print(test_output)

Feature: Netflix Distributed System Design

Scenario: Ensure high availability for streaming services at global scale
  Given the system is designed with a distributed architecture
  When a user requests to stream a movie
  Then the request is routed through the API Gateway Service (ZUUL)
  And the Play API controls the execution of micro-services
  And Hystrix commands prevent cascading failures
  And EV-Cache replicates data for faster access
  And the system ensures the availability of backend services and OCA servers

Scenario: Tackle network failures and system outages by resilience
  Given the system is designed with a distributed architecture
  When a network failure occurs
  Then Hystrix commands time out calls to micro-services
  And stop cascading errors
  And isolate points of failure from others
  And the system uses dynamic routing to switch to other nearby servers

Scenario: Minimize streaming latency
  Given the system is designed with a distributed architecture
  When a

In [38]:
documents = load_documents(
    r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
)

requirement_text = "\n\n".join(
    f"File: {doc.metadata.get('source', 'unknown')}\n"
    f"{doc.page_content.strip()}"
    for doc in documents
)

print(f"Aantal documenten/pagina's: {len(documents)}")
print(f"Aantal tekens in requirements: {len(requirement_text)}")
print(f"Aantal woorden ongeveer: {len(requirement_text.split())}")

Aantal documenten/pagina's: 65
Aantal tekens in requirements: 172235
Aantal woorden ongeveer: 26124


In [39]:
print(llm)

metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}} model='qwen2.5:7b' temperature=0.0


In [40]:
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate


@tool
def generate_test_cases() -> str:
    """Read requirements from PDF documents and generate at least ten BDD/Gherkin test scenarios."""

    # Load the PDF documents
    documents = load_documents(
        r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
    )

    # Combine all PDF content
    requirement_text = "\n\n".join(
        f"File: {doc.metadata.get('source', 'unknown')}\n"
        f"{doc.page_content.strip()}"
        for doc in documents
    )

    # Split the requirements into smaller chunks
    words = requirement_text.split()
    chunk_size = 4000

    chunks = [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Generate BDD/Gherkin test scenarios based ONLY on the requirements
provided below.

Generate several distinct scenarios from this requirement section.
Each scenario must test a different behavior, requirement, condition,
or system characteristic.

Do not invent requirements, business rules, limits, error messages,
or behavior that is not supported by the provided text.

Use this format:

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]

Requirements:
{requirements}

Return only the BDD/Gherkin scenarios.
""")

    all_scenarios = []

    # Generate scenarios for every chunk
    for index, chunk in enumerate(chunks, start=1):
        print(f"Processing requirement chunk {index}/{len(chunks)}...")

        prompt_value = prompt.invoke({
            "requirements": chunk
        })

        response = llm.invoke(prompt_value)

        all_scenarios.append(response.content)

    # Combine all generated scenarios
    final_output = "\n\n".join(all_scenarios)

    return f"""Feature: Netflix Distributed System Design

{final_output}
"""


print(generate_test_cases)

name='generate_test_cases' description='Read requirements from PDF documents and generate at least ten BDD/Gherkin test scenarios.' args_schema=<class 'langchain_core.utils.pydantic.generate_test_cases'> func=<function generate_test_cases at 0x0000015661F50AE0>


In [41]:
test_output = generate_test_cases.invoke({})

print(test_output)

Processing requirement chunk 1/7...
Processing requirement chunk 2/7...
Processing requirement chunk 3/7...
Processing requirement chunk 4/7...
Processing requirement chunk 5/7...
Processing requirement chunk 6/7...
Processing requirement chunk 7/7...
Feature: Netflix Distributed System Design

Based on the information provided, it seems there are no direct BDD/Gherkin scenarios present. The text appears to be a research methodology and results summary for a study on consumer well-being related to Netflix. However, if we were to infer a Gherkin scenario based on the research findings, it might look something like this:

```gherkin
Feature: Impact of Various Factors on Consumer Well-Being

  Scenario: Self-Image Congruence positively impacts consumer well-being
    Given a consumer with high self-image congruence
    When their self-image congruence is increased by 1%
    Then their consumer well-being is expected to increase by 0.466%

  Scenario: Convenience Factors have a negative im

In [ ]:
Dit werkt technisch, maar de output laat een belangrijk probleem zien: de 7 chunks worden wel verwerkt, maar Qwen genereert ook scenario's uit irrelevante PDF-inhoud. Je krijgt bijvoorbeeld scenario's over consumer well-being en recommender systems, terwijl je eigenlijk de requirements uit de betreffende PDF's wilt gebruiken.

De oorzaak is dat je map blijkbaar meerdere PDF's bevat en we nu alle 65 pagina's als requirements behandelen. De oorspronkelijke cursusopdracht lijkt juist bedoeld om de relevante requirement-PDF's te lezen.

We moeten daarom eerst bekijken welke PDF-bestanden precies in de Docs-map staan. Daarna kunnen we de tool alleen de juiste requirements laten gebruiken.

Waarom deze stap

We hoeven de LLM nu niet opnieuw aan te roepen. Eerst bepalen we welke bestanden er in de map zitten. Daarmee kunnen we voorkomen dat onderzoeksartikelen en andere documentatie als software requirements worden geïnterpreteerd.

Voer deze cel uit:

In [42]:
from pathlib import Path

docs_path = Path(
    r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs\Docs"
)

pdf_files = list(docs_path.glob("*.pdf"))

print(f"Aantal PDF-bestanden: {len(pdf_files)}")

for pdf in pdf_files:
    print(pdf.name)

Aantal PDF-bestanden: 3
Netflix_Consumer_WellBeing.pdf
Netflix_recommendation_system.pdf
Netflix_system_design.pdf


Dat verklaart de output. Je hebt drie PDF's, maar alleen Netflix_system_design.pdf is de relevante bron voor de BDD-scenario's uit deze les. De andere twee documenten gaan over consumer well-being en recommendation systems, waardoor het model daar ook scenario's uit genereert.

We moeten de tool dus niet alle PDF's laten inladen. Voor deze les laten we hem uitsluitend Netflix_system_design.pdf lezen.

Waarom deze stap

Dit sluit ook beter aan bij de cursus: de agent moet de requirements uit de relevante externe documentatie gebruiken, niet willekeurige informatie uit alle PDF's in dezelfde map.

Vervang je huidige generate_test_cases-functie volledig door deze versie:

In [43]:
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader


@tool
def generate_test_cases() -> str:
    """Read the Netflix system design requirements from PDF and generate BDD/Gherkin test scenarios."""

    # Load only the relevant requirements PDF
    pdf_path = (
        r"C:\Users\Mill\agentsforQA\.venv\Section3_WorkingWithExternalDocs"
        r"\Docs\Netflix_system_design.pdf"
    )

    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # Combine the PDF content
    requirement_text = "\n\n".join(
        doc.page_content.strip()
        for doc in documents
        if doc.page_content.strip()
    )

    prompt = PromptTemplate.from_template("""
You are a QA automation engineer.

Read the Netflix system design requirements below and generate
at least 10 separate BDD/Gherkin test scenarios.

IMPORTANT:
- Generate AT LEAST 10 scenarios.
- Each scenario must test a different behavior or requirement.
- Use only information explicitly supported by the document.
- Do not use information from outside the document.
- Do not invent requirements, business rules, error messages,
  limits, or expected behavior.
- Do not create scenarios about research methodology,
  consumer well-being, or recommendation systems unless they
  are explicitly described as system requirements in this document.

Use this format:

Feature: [feature name]

Scenario: [scenario name]
  Given [precondition]
  When [action]
  Then [expected result]

Requirements:
{requirements}

Return only the BDD/Gherkin scenarios.
""")

    prompt_value = prompt.invoke({
        "requirements": requirement_text
    })

    response = llm.invoke(prompt_value)

    return response.content


print(generate_test_cases)

name='generate_test_cases' description='Read the Netflix system design requirements from PDF and generate BDD/Gherkin test scenarios.' args_schema=<class 'langchain_core.utils.pydantic.generate_test_cases'> func=<function generate_test_cases at 0x0000015661EF7F60>


In [44]:
test_output = generate_test_cases.invoke({})

print(test_output)

### BDD/Gherkin Scenarios

```gherkin
Feature: Netflix Distributed System Architecture

  Scenario: Ensuring High Availability for Streaming Services
    Given the system is designed to handle global scale
    And the availability of streaming services depends on backend services and OCA servers
    When a request is made to a streaming service
    Then the load balancer should prevent overloading workloads
    And the API Gateway Service should allow dynamic routing
    And the Play API should control the execution of microservices and prevent cascading failures
    And the EV-Cache should replicate data for faster access
    And the system should handle 30 million requests a day with linear scalability and milliseconds latency

  Scenario: Tackling Network Failures and System Outages by Resilience
    Given the system is designed to recover from errors or interruptions
    When a network failure or system outage occurs
    Then the system should use Hystrix commands to timeout calls 

Dit is nu inhoudelijk veel beter. De tool leest de juiste PDF en de irrelevante onderwerpen zijn verdwenen.

Maar er is nog één punt: ondanks de instructie om minimaal 10 scenario's te genereren, geeft Qwen opnieuw maar 4 scenario's. Dat is een modelgedragsprobleem, niet meer een probleem met je PDF-loader.

Voor de cursus zou ik hier nu niet verder aan blijven sleutelen. De kern van de les is namelijk gelukt:

PDF → requirements lezen → tool aanroepen → LLM → BDD/Gherkin.

We moeten nu terug naar de agent, want dat is het eigenlijke doel van deze les. Je agent is eerder al correct aangemaakt met:

In [45]:
agent = create_agent(
    model=llm,
    tools=[generate_test_cases]
)

In [46]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Generate BDD/Gherkin test scenarios based on the requirements in the PDF document."
        }
    ]
})

print(result["messages"][-1].content)

Based on the requirements in the provided PDF document, here are the BDD/Gherkin test scenarios for the Netflix streaming service architecture:

```gherkin
Feature: Netflix Streaming Service Architecture
  Scenario: Ensuring High Availability for Streaming Services
    Given the system is designed with multiple microservices
    And there are load balancers to distribute requests
    And API Gateway Service (ZUUL) for dynamic routing
    And Play API for controlling microservices execution
    And Hystrix for preventing cascading failures
    And EV-Cache for data replication and faster access
    When a user requests to stream a movie
    Then the system should handle the request without downtime
    And the availability of streaming services should be maintained
    And the user should be able to watch the movie without interruptions

  Scenario: Tackling Network Failures and System Outages by Resilience
    Given the system is designed with Hystrix commands
    And there are fallbac

Ja. Dit resultaat laat zien dat de volledige agent-keten nu werkt.
De keten is nu:
User prompt → Agent → generate_test_cases tool → Netflix_system_design.pdf → LLM → BDD/Gherkin
Je hebt nu ook geen NameError meer en de agent gebruikt daadwerkelijk de nieuwe PDF-tool.
Wel belangrijk: de cursus vroeg om het mechanisme te demonstreren, niet per se om precies 10 scenario's af te dwingen. Jouw Qwen-model genereert hier 4 scenario's, maar de vier belangrijkste gebieden uit de PDF zijn wel afgedekt:
High availability
Network failures / resilience
Streaming latency
Scalability
Ik zou de code op dit punt niet verder aanpassen. De oorspronkelijke cursusfunctionaliteit is bereikt.
Als je verder wilt met de volgende les, plak de transcriptie/code daarvan hier; dan bouwen we vanaf deze werkende situatie verder.

Stap 1: controleer je virtuele omgeving
Open PowerShell en ga naar je bestaande project:
cd C:\Users\Mill\agentsforQA
Controleer daarna of je .venv312 actief is. Je prompt zou ongeveer zo moeten beginnen:
(.venv312) PS C:\Users\Mill\agentsforQA>
Waarom deze stap
De docent installeert AutoGen Studio vanuit een virtual environment. Jij hebt al .venv312, dus we gebruiken dezelfde omgeving en maken niet opnieuw een Python-omgeving aan.
Voer alleen deze stap uit en stuur mij wat PowerShell laat zien. Daarna doen we de installatie van AutoGen Studio.

Stap 2: installeer AutoGen Studio
Voer nu alleen dit commando uit:
pip install -U autogenstudio
Waarom deze stap
Dit is het pakket dat de docent in deze les gebruikt om AutoGen Studio lokaal te starten. We installeren het in jouw bestaande .venv312, zodat het gescheiden blijft van je andere Python-installaties.
Laat de installatie volledig afronden en stuur daarna de laatste regels van de output. Dan bepalen we de volgende stap.

De installatie is geslaagd. Die laatste regels zijn de lijst met geïnstalleerde pakketten; er staat geen foutmelding tussen.

Stap 3: controleer AutoGen Studio

Voer nu alleen dit uit:

autogenstudio --help
Waarom deze stap

Hiermee controleren we of het autogenstudio-commando correct beschikbaar is vanuit jouw actieve .venv312. Als dit werkt, starten we daarna pas de Studio, net zoals in de video.

Stuur de output hiervan door.

In [47]:
autogenstudio --help

NameError: name 'autogenstudio' is not defined